# ARC-AGI-3 — Hybrid Explorer Agent (self-contained)

General, training-free interactive agent (MIT-0): perception (object segmentation,
counter/distractor masking) -> state-transition graph exploration -> motion-model avatar
navigation -> 5-tier click salience. The arcagi3 package is embedded in this notebook
(written to /kaggle/working) so there is no external dependency; the agent is fail-safe
(random fallback) so it always acts.


In [ ]:
# Install the ARC-AGI-3 toolkit + engine offline from the competition wheels.
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
# Phase A.0 probe: record torch/GPU availability on Kaggle (go/no-go for online learning).
# Runs UNCONDITIONALLY (interactive run too) because code-competition RERUN logs are hidden
# — the interactive kernel log IS readable via `kaggle kernels output`, and the interactive
# GPU image is a strong proxy for the rerun image. Fully try/excepted; never crashes the run.
import os as _os
print('=== ARCAGI3_EVAL_PROBE_BEGIN ===', flush=True)
print('is_rerun', bool(_os.getenv('KAGGLE_IS_COMPETITION_RERUN')), flush=True)
try:
    import numpy as _np; print('numpy', _np.__version__, flush=True)
except Exception as _e:
    print('numpy import FAILED', repr(_e), flush=True)
try:
    import torch as _t
    print('torch', _t.__version__, flush=True)
    print('cuda_available', _t.cuda.is_available(), flush=True)
    print('device_count', _t.cuda.device_count(), flush=True)
    for _i in range(_t.cuda.device_count()):
        _p = _t.cuda.get_device_properties(_i)
        print(f'gpu{_i}', _p.name, round(_p.total_memory/1e9, 2), 'GB', flush=True)
except Exception as _e:
    print('torch import FAILED', repr(_e), flush=True)
try:
    _w = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
    print('torch_wheels', [f for f in _os.listdir(_w) if 'torch' in f.lower()], flush=True)
except Exception as _e:
    print('wheels listdir FAILED', repr(_e), flush=True)
print('=== ARCAGI3_EVAL_PROBE_END ===', flush=True)


In [ ]:
# Write the self-contained arcagi3 package to /kaggle/working/arcagi3 (no dataset dep).
import base64, os, pathlib
os.makedirs('/kaggle/working/arcagi3', exist_ok=True)
PKG = {'__init__.py': 'IiIiQVJDLUFHSS0zIGNvbXBldGl0aW9uIGFnZW50IChBUkMgUHJpemUgMjAyNikuIiIiCg==', 'perception.py': 'IiIiUGVyY2VwdGlvbjogdHVybiBhIHJhdyBBUkMtQUdJLTMgZnJhbWUgaW50byBhbiBvYmplY3QtY2VudHJpYyBzdGF0ZS4KCkEgZnJhbWUgZnJvbSB0aGUgZW5naW5lIGlzIGFuIGludDggYXJyYXkgb2Ygc2hhcGUgKE4sIDY0LCA2NCkgaG9sZGluZyBvbmUgb3IgbW9yZQpzdWItZnJhbWVzIChhbmltYXRpb24vdHJhbnNpdGlvbiBzdGVwcykgd2l0aCBjb2xvciB2YWx1ZXMgMC0xNS4gVGhlIGFnZW50IHJlYXNvbnMgb3Zlcgp0aGUgZmluYWwgc2V0dGxlZCBzdWItZnJhbWUgcGx1cyBhIHRlbXBvcmFsbHktZGVyaXZlZCBtYXNrIG9mICJ2b2xhdGlsZSIgY2VsbHMgKHN0YXR1cwpiYXJzIC8gY291bnRlcnMpIHRoYXQgbXVzdCBiZSBpZ25vcmVkIHdoZW4gZGVjaWRpbmcgd2hldGhlciB0d28gc3RhdGVzIGFyZSB0aGUgc2FtZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gZnVuY3Rvb2xzIGltcG9ydCBscnVfY2FjaGUKCmltcG9ydCBudW1weSBhcyBucAoKR1JJRCA9IDY0CgoKZGVmIHRvX2dyaWQoZnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gdGhlIGZpbmFsIHNldHRsZWQgNjR4NjQgc3ViLWZyYW1lIGFzIGFuIGludDggbmRhcnJheS4KCiAgICBBY2NlcHRzIGEgRnJhbWVEYXRhLCBhIGxpc3QsIG9yIGFuIG5kYXJyYXkgb2Ygc2hhcGUgKE4sNjQsNjQpIC8gKDY0LDY0KS4KICAgICIiIgogICAgaWYgaGFzYXR0cihmcmFtZSwgImZyYW1lIik6ICAjIGEgRnJhbWVEYXRhCiAgICAgICAgZnJhbWUgPSBmcmFtZS5mcmFtZQogICAgYXJyID0gbnAuYXNhcnJheShmcmFtZSwgZHR5cGU9bnAuaW50OCkKICAgIGlmIGFyci5uZGltID09IDM6CiAgICAgICAgYXJyID0gYXJyWy0xXQogICAgaWYgYXJyLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5leHBlY3RlZCBmcmFtZSBzaGFwZSB7YXJyLnNoYXBlfSIpCiAgICByZXR1cm4gYXJyCgoKZGVmIGdyaWRfc3RhY2soZnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gYWxsIHN1Yi1mcmFtZXMgYXMgKE4sNjQsNjQpOyBhbmltYXRpb24gYWNyb3NzIGEgc2luZ2xlIHN0ZXAuIiIiCiAgICBpZiBoYXNhdHRyKGZyYW1lLCAiZnJhbWUiKToKICAgICAgICBmcmFtZSA9IGZyYW1lLmZyYW1lCiAgICBhcnIgPSBucC5hc2FycmF5KGZyYW1lLCBkdHlwZT1ucC5pbnQ4KQogICAgaWYgYXJyLm5kaW0gPT0gMjoKICAgICAgICBhcnIgPSBhcnJbTm9uZSwgOiwgOl0KICAgIHJldHVybiBhcnIKCgpkZWYgZGV0ZWN0X2JhY2tncm91bmQoZ3JpZDogbnAubmRhcnJheSkgLT4gaW50OgogICAgIiIiTW9zdCBmcmVxdWVudCBjb2xvciA9IHByZXN1bWVkIGJhY2tncm91bmQuIiIiCiAgICB2YWxzLCBjb3VudHMgPSBucC51bmlxdWUoZ3JpZCwgcmV0dXJuX2NvdW50cz1UcnVlKQogICAgcmV0dXJuIGludCh2YWxzW2ludChucC5hcmdtYXgoY291bnRzKSldKQoKCmRlZiBlbmNvZGVfb25laG90KGdyaWQ6IG5wLm5kYXJyYXksIG51bV9jb2xvcnM6IGludCA9IDE2KSAtPiBucC5uZGFycmF5OgogICAgIiIiKEgsVykgaW50IGNvbG9yIGdyaWQgLT4gKG51bV9jb2xvcnMsSCxXKSBmbG9hdDMyIG9uZS1ob3QgKGZvciB0aGUgUGhhc2UgQSBDTk4pLiIiIgogICAgZyA9IG5wLmNsaXAobnAuYXNhcnJheShncmlkKSwgMCwgbnVtX2NvbG9ycyAtIDEpLmFzdHlwZShucC5pbnQ2NCkKICAgIG9oID0gbnAuemVyb3MoKG51bV9jb2xvcnMsIGcuc2hhcGVbMF0sIGcuc2hhcGVbMV0pLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgbnAucHV0X2Fsb25nX2F4aXMob2gucmVzaGFwZShudW1fY29sb3JzLCAtMSksCiAgICAgICAgICAgICAgICAgICAgICBnLnJlc2hhcGUoMSwgLTEpLCAxLjAsIGF4aXM9MCkKICAgIHJldHVybiBvaAoKCkBkYXRhY2xhc3MKY2xhc3MgT2JqOgogICAgIiIiQSBjb25uZWN0ZWQgcmVnaW9uIG9mIGEgc2luZ2xlIGNvbG9yICg0LWNvbm5lY3Rpdml0eSkuIiIiCgogICAgY29sb3I6IGludAogICAgY2VsbHM6IHR1cGxlW3R1cGxlW2ludCwgaW50XSwgLi4uXSAgIyAocm93LCBjb2wpIHBhaXJzCiAgICBiYm94OiB0dXBsZVtpbnQsIGludCwgaW50LCBpbnRdICAjIChyMCwgYzAsIHIxLCBjMSkgaW5jbHVzaXZlCiAgICBzaXplOiBpbnQKICAgIGNlbnRyb2lkOiB0dXBsZVtmbG9hdCwgZmxvYXRdCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG9wX2xlZnQoc2VsZikgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHJldHVybiAoc2VsZi5iYm94WzBdLCBzZWxmLmJib3hbMV0pCgogICAgQHByb3BlcnR5CiAgICBkZWYgd2lkdGgoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLmJib3hbM10gLSBzZWxmLmJib3hbMV0gKyAxCgogICAgQHByb3BlcnR5CiAgICBkZWYgaGVpZ2h0KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5iYm94WzJdIC0gc2VsZi5iYm94WzBdICsgMQoKCmRlZiBjb25uZWN0ZWRfY29tcG9uZW50cygKICAgIGdyaWQ6IG5wLm5kYXJyYXksCiAgICBiYWNrZ3JvdW5kOiBpbnQgfCBOb25lID0gTm9uZSwKICAgIGluY2x1ZGVfYmFja2dyb3VuZDogYm9vbCA9IEZhbHNlLAopIC0+IGxpc3RbT2JqXToKICAgICIiIjQtY29ubmVjdGl2aXR5IGNvbm5lY3RlZCBjb21wb25lbnRzIG9mIGVxdWFsIGNvbG9yIChtZW1vaXplZCBwZXIgZ3JpZCkuCgogICAgQmFja2dyb3VuZCBjb2xvciBjb21wb25lbnRzIGFyZSBza2lwcGVkIHVubGVzcyBpbmNsdWRlX2JhY2tncm91bmQgaXMgVHJ1ZS4gUmVzdWx0cyBhcmUKICAgIGNhY2hlZCBvbiB0aGUgcmF3IGdyaWQgYnl0ZXM6IGEgc2luZ2xlIGRlY2lzaW9uIHN0ZXAgY2FsbHMgdGhpcyBzZXZlcmFsIHRpbWVzIG9uIHRoZQogICAgU0FNRSBncmlkIChzdGF0ZSBoYXNoaW5nLCBjbGljayB0YXJnZXRzLCBuYXYgdGFyZ2V0aW5nKSwgc28gbWVtb2l6aW5nIGlzIGEgcHVyZQogICAgc3BlZWR1cCAoaWRlbnRpY2FsIHJlc3VsdHMpIHRoYXQgYnV5cyBtb3JlIGFjdGlvbnMvc2VjIOKAlCBpLmUuIG1vcmUgbGV2ZWxzIGF0IGV2YWwuCiAgICAiIiIKICAgIGlmIGJhY2tncm91bmQgaXMgTm9uZToKICAgICAgICBiYWNrZ3JvdW5kID0gZGV0ZWN0X2JhY2tncm91bmQoZ3JpZCkKICAgIHJldHVybiBfY29ubmVjdGVkX2NvbXBvbmVudHNfY2FjaGVkKAogICAgICAgIG5wLmFzY29udGlndW91c2FycmF5KGdyaWQpLnRvYnl0ZXMoKSwgZ3JpZC5zaGFwZSwgaW50KGJhY2tncm91bmQpLCBpbmNsdWRlX2JhY2tncm91bmQKICAgICkKCgpAbHJ1X2NhY2hlKG1heHNpemU9MTYpCmRlZiBfY29ubmVjdGVkX2NvbXBvbmVudHNfY2FjaGVkKGdyaWRfYnl0ZXMsIHNoYXBlLCBiYWNrZ3JvdW5kLCBpbmNsdWRlX2JhY2tncm91bmQpIC0+IGxpc3RbT2JqXToKICAgIGdyaWQgPSBucC5mcm9tYnVmZmVyKGdyaWRfYnl0ZXMsIGR0eXBlPW5wLmludDgpLnJlc2hhcGUoc2hhcGUpCiAgICBoLCB3ID0gc2hhcGUKICAgIHNlZW4gPSBucC56ZXJvcygoaCwgdyksIGR0eXBlPWJvb2wpCiAgICBvYmpzOiBsaXN0W09ial0gPSBbXQogICAgZm9yIHIgaW4gcmFuZ2UoaCk6CiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CiAgICAgICAgICAgIGlmIHNlZW5bciwgY106CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBjb2xvciA9IGludChncmlkW3IsIGNdKQogICAgICAgICAgICBpZiBub3QgaW5jbHVkZV9iYWNrZ3JvdW5kIGFuZCBjb2xvciA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICAgICAgc2VlbltyLCBjXSA9IFRydWUKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgQkZTIGZsb29kIGZpbGwKICAgICAgICAgICAgY2VsbHM6IGxpc3RbdHVwbGVbaW50LCBpbnRdXSA9IFtdCiAgICAgICAgICAgIHEgPSBkZXF1ZShbKHIsIGMpXSkKICAgICAgICAgICAgc2VlbltyLCBjXSA9IFRydWUKICAgICAgICAgICAgcjAgPSByMSA9IHIKICAgICAgICAgICAgYzAgPSBjMSA9IGMKICAgICAgICAgICAgc3IgPSBzYyA9IDAKICAgICAgICAgICAgd2hpbGUgcToKICAgICAgICAgICAgICAgIGNyLCBjYyA9IHEucG9wbGVmdCgpCiAgICAgICAgICAgICAgICBjZWxscy5hcHBlbmQoKGNyLCBjYykpCiAgICAgICAgICAgICAgICBzciArPSBjcgogICAgICAgICAgICAgICAgc2MgKz0gY2MKICAgICAgICAgICAgICAgIHIwLCByMSA9IG1pbihyMCwgY3IpLCBtYXgocjEsIGNyKQogICAgICAgICAgICAgICAgYzAsIGMxID0gbWluKGMwLCBjYyksIG1heChjMSwgY2MpCiAgICAgICAgICAgICAgICBmb3IgZHIsIGRjIGluICgoMSwgMCksICgtMSwgMCksICgwLCAxKSwgKDAsIC0xKSk6CiAgICAgICAgICAgICAgICAgICAgbnIsIG5jID0gY3IgKyBkciwgY2MgKyBkYwogICAgICAgICAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoIGFuZCAwIDw9IG5jIDwgdyBhbmQgbm90IHNlZW5bbnIsIG5jXSBhbmQgaW50KGdyaWRbbnIsIG5jXSkgPT0gY29sb3I6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlZW5bbnIsIG5jXSA9IFRydWUKICAgICAgICAgICAgICAgICAgICAgICAgcS5hcHBlbmQoKG5yLCBuYykpCiAgICAgICAgICAgIG4gPSBsZW4oY2VsbHMpCiAgICAgICAgICAgIG9ianMuYXBwZW5kKAogICAgICAgICAgICAgICAgT2JqKAogICAgICAgICAgICAgICAgICAgIGNvbG9yPWNvbG9yLAogICAgICAgICAgICAgICAgICAgIGNlbGxzPXR1cGxlKGNlbGxzKSwKICAgICAgICAgICAgICAgICAgICBiYm94PShyMCwgYzAsIHIxLCBjMSksCiAgICAgICAgICAgICAgICAgICAgc2l6ZT1uLAogICAgICAgICAgICAgICAgICAgIGNlbnRyb2lkPShzciAvIG4sIHNjIC8gbiksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgIHJldHVybiBvYmpzCgoKZGVmIHN0YXRlX2hhc2goZ3JpZDogbnAubmRhcnJheSwgbWFzazogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lKSAtPiBieXRlczoKICAgICIiIkV4YWN0IGhhc2ggb2YgdGhlIGdyaWQsIG9wdGlvbmFsbHkgemVyb2luZyBtYXNrZWQgKHZvbGF0aWxlKSBjZWxscyBmaXJzdC4iIiIKICAgIGlmIG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgZyA9IGdyaWQuY29weSgpCiAgICAgICAgZ1ttYXNrXSA9IC0xCiAgICAgICAgcmV0dXJuIGcudG9ieXRlcygpCiAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoZ3JpZCkudG9ieXRlcygpCgoKZGVmIF9vYmplY3RfdHVwbGVzKG9ianM6IGxpc3RbT2JqXSwgaWdub3JlX2NvbG9yczogc2V0W2ludF0gfCBOb25lID0gTm9uZSkgLT4gbGlzdFt0dXBsZV06CiAgICAiIiJTb3J0ZWQgWyhjb2xvciwgcjAsIGMwLCByMSwgYzEsIHNpemUpXSBzdW1tYXJ5IG9mIGNvbm5lY3RlZCBjb21wb25lbnRzLgoKICAgIFNpbmdsZSBzb3VyY2Ugb2YgdHJ1dGggZm9yIHRoZSBvYmplY3QtbGV2ZWwgc3RhdGUgc3VtbWFyeSwgc2hhcmVkIGJ5CiAgICBgYG9iamVjdF9zdGF0ZV9rZXlgYCBhbmQgYGBmb3J3YXJkX21vZGVsLlNjZW5lLmtleWBgIHNvIHRoZSB0d28gYXJlIGJ5dGUtaWRlbnRpY2FsCiAgICAodGhlIEM0IGxpbmNocGluOiBhIGNvcnJlY3QgcHJlZGljdGlvbidzIGtleSBtdXN0IGVxdWFsIHRoZSByZWFsIG5leHQga2V5IGV4YWN0bHkpLgogICAgIiIiCiAgICBpZ25vcmUgPSBpZ25vcmVfY29sb3JzIG9yIHNldCgpCiAgICBwYXJ0cyA9IFtdCiAgICBmb3IgbyBpbiBvYmpzOgogICAgICAgIGlmIG8uY29sb3IgaW4gaWdub3JlOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHIwLCBjMCwgcjEsIGMxID0gby5iYm94CiAgICAgICAgcGFydHMuYXBwZW5kKChvLmNvbG9yLCByMCwgYzAsIHIxLCBjMSwgby5zaXplKSkKICAgIHBhcnRzLnNvcnQoKQogICAgcmV0dXJuIHBhcnRzCgoKZGVmIG9iamVjdF9zdGF0ZV9rZXkoZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgIGlnbm9yZV9jb2xvcnM6IHNldFtpbnRdIHwgTm9uZSA9IE5vbmUpIC0+IGJ5dGVzOgogICAgIiIiQ29hcnNlLCByb2J1c3Qgc3RhdGUga2V5IGZyb20gT0JKRUNUIHN0cnVjdHVyZSAobm90IHJhdyBwaXhlbHMpLgoKICAgIEVhY2ggbm9uLWJhY2tncm91bmQsIG5vbi1pZ25vcmVkIGNvbm5lY3RlZCBjb21wb25lbnQgaXMgc3VtbWFyaXNlZCBhcwogICAgKGNvbG9yLCByMCwgYzAsIHIxLCBjMSwgc2l6ZSkuIFNvcnRpbmcgKyBzZXJpYWxpc2luZyB0aGVzZSBpcyBmYXIgbW9yZSBzdGFibGUgdGhhbiBhCiAgICBwaXhlbCBoYXNoOiBpdCBjb2xsYXBzZXMgd2l0aGluLW9iamVjdCBqaXR0ZXIgYW5kIGlycmVsZXZhbnQgc2luZ2xlLXBpeGVsIG5vaXNlIHRoYXQKICAgIHdvdWxkIG90aGVyd2lzZSBleHBsb2RlIHRoZSBzdGF0ZSBncmFwaCBvbiByZWFsIGdhbWVzLCB3aGlsZSBzdGlsbCBkaXN0aW5ndWlzaGluZwogICAgb2JqZWN0IG1vdmVzLCBhcHBlYXJhbmNlcy9kaXNhcHBlYXJhbmNlcywgYW5kIHNoYXBlIGNoYW5nZXMuIE1hdGNoZXMgdGhlIFNPVEEncwogICAgb2JqZWN0LXNlZ21lbnRhdGlvbiBhcHByb2FjaC4KICAgICIiIgogICAgaWYgYmFja2dyb3VuZCBpcyBOb25lOgogICAgICAgIGJhY2tncm91bmQgPSBkZXRlY3RfYmFja2dyb3VuZChncmlkKQogICAgcGFydHMgPSBfb2JqZWN0X3R1cGxlcyhjb25uZWN0ZWRfY29tcG9uZW50cyhncmlkLCBiYWNrZ3JvdW5kPWJhY2tncm91bmQpLCBpZ25vcmVfY29sb3JzKQogICAgcmV0dXJuIHJlcHIocGFydHMpLmVuY29kZSgpCgoKY2xhc3MgVm9sYXRpbGl0eVRyYWNrZXI6CiAgICAiIiJUcmFja3Mgd2hpY2ggY2VsbHMgY2hhbmdlIGZyZXF1ZW50bHkgYWNyb3NzIHN0ZXBzIHRvIG1hc2sgc3RhdHVzIGJhcnMvY291bnRlcnMuCgogICAgQ2VsbHMgdGhhdCBjaGFuZ2Ugb24gKGFsbW9zdCkgZXZlcnkgc3RlcCByZWdhcmRsZXNzIG9mIGVmZmVjdCBhcmUgbGlrZWx5IHN0ZXAKICAgIGNvdW50ZXJzIG9yIGFuaW1hdGVkIGRlY29yYXRpb25zIGFuZCBzaG91bGQgYmUgZXhjbHVkZWQgZnJvbSB0aGUgc3RhdGUga2V5IHNvIHRoZQogICAgc3RhdGUgZ3JhcGggZG9lc24ndCBleHBsb2RlLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRocmVzaG9sZDogZmxvYXQgPSAwLjksIG1pbl9zdGVwczogaW50ID0gOCkgLT4gTm9uZToKICAgICAgICBzZWxmLnRocmVzaG9sZCA9IHRocmVzaG9sZAogICAgICAgIHNlbGYubWluX3N0ZXBzID0gbWluX3N0ZXBzCiAgICAgICAgc2VsZi5jaGFuZ2VzOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUgICMgbGF6aWx5IHNpemVkIHRvIHRoZSBhY3R1YWwgZnJhbWUKICAgICAgICBzZWxmLnNoYXBlOiB0dXBsZVtpbnQsIGludF0gPSAoR1JJRCwgR1JJRCkKICAgICAgICBzZWxmLnN0ZXBzID0gMAogICAgICAgIHNlbGYuX3ByZXY6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZQoKICAgIGRlZiB1cGRhdGUoc2VsZiwgZ3JpZDogbnAubmRhcnJheSkgLT4gTm9uZToKICAgICAgICAjIExhemlseSBhZG9wdCB0aGUgcmVhbCBmcmFtZSBzaGFwZTsgcmVzZXQgaWYgaXQgZXZlciBjaGFuZ2VzIChkZWZlbnNpdmUpLgogICAgICAgIGlmIHNlbGYuY2hhbmdlcyBpcyBOb25lIG9yIGdyaWQuc2hhcGUgIT0gc2VsZi5zaGFwZToKICAgICAgICAgICAgc2VsZi5zaGFwZSA9IGdyaWQuc2hhcGUKICAgICAgICAgICAgc2VsZi5jaGFuZ2VzID0gbnAuemVyb3Moc2VsZi5zaGFwZSwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgICAgIHNlbGYuc3RlcHMgPSAwCiAgICAgICAgICAgIHNlbGYuX3ByZXYgPSBOb25lCiAgICAgICAgaWYgc2VsZi5fcHJldiBpcyBub3QgTm9uZSBhbmQgc2VsZi5fcHJldi5zaGFwZSA9PSBncmlkLnNoYXBlOgogICAgICAgICAgICBzZWxmLmNoYW5nZXMgKz0gKGdyaWQgIT0gc2VsZi5fcHJldikuYXN0eXBlKG5wLmludDMyKQogICAgICAgICAgICBzZWxmLnN0ZXBzICs9IDEKICAgICAgICBzZWxmLl9wcmV2ID0gZ3JpZC5jb3B5KCkKCiAgICBkZWYgbWFzayhzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIkJvb2xlYW4gbWFzayBvZiBjZWxscyB0byBpZ25vcmUgKFRydWUgPSB2b2xhdGlsZSkuIiIiCiAgICAgICAgaWYgc2VsZi5jaGFuZ2VzIGlzIE5vbmUgb3Igc2VsZi5zdGVwcyA8IHNlbGYubWluX3N0ZXBzOgogICAgICAgICAgICByZXR1cm4gbnAuemVyb3Moc2VsZi5zaGFwZSwgZHR5cGU9Ym9vbCkKICAgICAgICByZXR1cm4gKHNlbGYuY2hhbmdlcyAvIG1heChzZWxmLnN0ZXBzLCAxKSkgPj0gc2VsZi50aHJlc2hvbGQKCgpkZWYgc2FsaWVudF9jbGlja190YXJnZXRzKAogICAgZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50IHwgTm9uZSA9IE5vbmUsIG1heF90YXJnZXRzOiBpbnQgPSA2NCwKICAgIGNvYXJzZV9ncmlkX3N0ZXA6IGludCA9IDAsCikgLT4gbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV06CiAgICAiIiJQcm9wb3NlICh4LCB5LCBwcmlvcml0eSkgY2xpY2sgdGFyZ2V0cyBmcm9tIG9iamVjdCBnZW9tZXRyeS4KCiAgICBPYmplY3QtY2VudHJpYyBpbnN0ZWFkIG9mIGJydXRlLWZvcmNpbmcgYWxsIDQwOTYgcGl4ZWxzLiBQcmlvcml0eSBpcyBhIHNhbGllbmNlCiAgICB0aWVyIChsb3dlciA9IHRyeSBmaXJzdCk6IHNtYWxsIGRpc3RpbmN0IG9iamVjdHMgYW5kIHRoZWlyIGNvcm5lcnMgYXJlIG1vc3QgbGlrZWx5CiAgICBpbnRlcmFjdGl2ZS4gUmV0dXJucyAoeD1jb2wsIHk9cm93LCBwcmlvcml0eSkuCgogICAgSWYgY29hcnNlX2dyaWRfc3RlcCA+IDAsIGFsc28gYWRkIGEgY29hcnNlIGxhdHRpY2Ugb2YgbG93LXByaW9yaXR5IGZhbGxiYWNrIHRhcmdldHMKICAgIChldmVyeSBgY29hcnNlX2dyaWRfc3RlcGAgcGl4ZWxzKSBzbyBsYXJnZSBjbGljayBhY3Rpb24tc3BhY2VzIChlLmcuIGZ0MDkncyB+NDA5NgogICAgcG9zaXRpb25zKSB3aGVyZSB0aGUgZ29hbCBjZWxsIGlzbid0IGFuIG9iamVjdCBjZW50cm9pZCBhcmUgc3RpbGwgcmVhY2hhYmxlLgogICAgIiIiCiAgICBpZiBiYWNrZ3JvdW5kIGlzIE5vbmU6CiAgICAgICAgYmFja2dyb3VuZCA9IGRldGVjdF9iYWNrZ3JvdW5kKGdyaWQpCiAgICBoLCB3ID0gZ3JpZC5zaGFwZQogICAgb2JqcyA9IGNvbm5lY3RlZF9jb21wb25lbnRzKGdyaWQsIGJhY2tncm91bmQ9YmFja2dyb3VuZCkKICAgICMgY29sb3IgcmFyaXR5OiByYXJlciBjb2xvcnMgYXJlIG1vcmUgbGlrZWx5IGludGVyYWN0aXZlIChidXR0b25zL2l0ZW1zKQogICAgY29sb3JfY291bnRzOiBkaWN0W2ludCwgaW50XSA9IHt9CiAgICBmb3IgbyBpbiBvYmpzOgogICAgICAgIGNvbG9yX2NvdW50c1tvLmNvbG9yXSA9IGNvbG9yX2NvdW50cy5nZXQoby5jb2xvciwgMCkgKyAxCiAgICB0YXJnZXRzOiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXSA9IFtdCiAgICBmb3IgbyBpbiBvYmpzOgogICAgICAgIHIsIGMgPSBvLmNlbnRyb2lkCiAgICAgICAgY3IsIGNjID0gaW50KHJvdW5kKHIpKSwgaW50KHJvdW5kKGMpKQogICAgICAgIHIwLCBjMCwgcjEsIGMxID0gby5iYm94CiAgICAgICAgIyBTT1RBLXN0eWxlIDUgc2FsaWVuY2UgdGllcnMgKGxvd2VyID0gdHJ5IGZpcnN0KTogc21hbGwgKyByYXJlLWNvbG9yIG9iamVjdHMgYXJlCiAgICAgICAgIyB0aGUgbW9zdCBsaWtlbHkgaW50ZXJhY3RpdmUgZWxlbWVudHM7IHdpZGUgZmxhdCBlZGdlLWh1Z2dpbmcgYmxvYnMgKHN0YXR1cyBiYXJzKQogICAgICAgICMgZ28gbGFzdC4KICAgICAgICBpc19zdGF0dXNfYmFyID0gKG8uaGVpZ2h0IDw9IDIgb3Igby53aWR0aCA8PSAyKSBhbmQgKG8ud2lkdGggPj0gdyAqIDAuNiBvciBvLmhlaWdodCA+PSBoICogMC42KQogICAgICAgIHJhcmUgPSBjb2xvcl9jb3VudHMuZ2V0KG8uY29sb3IsIDkpIDw9IDIKICAgICAgICBpZiBpc19zdGF0dXNfYmFyOgogICAgICAgICAgICBwcmlvID0gNAogICAgICAgIGVsaWYgby5zaXplIDw9IDQ6CiAgICAgICAgICAgIHByaW8gPSAwIGlmIHJhcmUgZWxzZSAxCiAgICAgICAgZWxpZiBvLnNpemUgPD0gMTY6CiAgICAgICAgICAgIHByaW8gPSAxIGlmIHJhcmUgZWxzZSAyCiAgICAgICAgZWxpZiBvLnNpemUgPD0gNjQ6CiAgICAgICAgICAgIHByaW8gPSAyIGlmIHJhcmUgZWxzZSAzCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbyA9IDMKICAgICAgICB0YXJnZXRzLmFwcGVuZCgoY2MsIGNyLCBwcmlvKSkKICAgICAgICAjIGNvcm5lcnMgb2YgbGFyZ2VyIG9iamVjdHMgKGhhbmRsZXMvZWRnZXMpLCBvbmUgdGllciBsb3dlcgogICAgICAgIGlmIG8uc2l6ZSA+IDggYW5kIG5vdCBpc19zdGF0dXNfYmFyOgogICAgICAgICAgICBmb3IgKHl5LCB4eCkgaW4gKChyMCwgYzApLCAocjAsIGMxKSwgKHIxLCBjMCksIChyMSwgYzEpKToKICAgICAgICAgICAgICAgIHRhcmdldHMuYXBwZW5kKCh4eCwgeXksIHByaW8gKyAxKSkKICAgICMgY29hcnNlIGxhdHRpY2UgZmFsbGJhY2sgZm9yIGxhcmdlIGNsaWNrIHNwYWNlcyAobG93ZXN0IHByaW9yaXR5KQogICAgaWYgY29hcnNlX2dyaWRfc3RlcCBhbmQgY29hcnNlX2dyaWRfc3RlcCA+IDA6CiAgICAgICAgaCwgdyA9IGdyaWQuc2hhcGUKICAgICAgICBvZmYgPSBjb2Fyc2VfZ3JpZF9zdGVwIC8vIDIKICAgICAgICBmb3IgeXkgaW4gcmFuZ2Uob2ZmLCBoLCBjb2Fyc2VfZ3JpZF9zdGVwKToKICAgICAgICAgICAgZm9yIHh4IGluIHJhbmdlKG9mZiwgdywgY29hcnNlX2dyaWRfc3RlcCk6CiAgICAgICAgICAgICAgICB0YXJnZXRzLmFwcGVuZCgoeHgsIHl5LCA5KSkKICAgICMgZGVkdXAga2VlcGluZyBiZXN0IChsb3dlc3QpIHByaW9yaXR5CiAgICBiZXN0OiBkaWN0W3R1cGxlW2ludCwgaW50XSwgaW50XSA9IHt9CiAgICBmb3IgeCwgeSwgcCBpbiB0YXJnZXRzOgogICAgICAgIGsgPSAoeCwgeSkKICAgICAgICBpZiBrIG5vdCBpbiBiZXN0IG9yIHAgPCBiZXN0W2tdOgogICAgICAgICAgICBiZXN0W2tdID0gcAogICAgb3V0ID0gWyh4LCB5LCBwKSBmb3IgKHgsIHkpLCBwIGluIGJlc3QuaXRlbXMoKV0KICAgIG91dC5zb3J0KGtleT1sYW1iZGEgdDogdFsyXSkKICAgIHJldHVybiBvdXRbOm1heF90YXJnZXRzXQo=', 'world_model.py': 'IiIiV29ybGQgbW9kZWw6IGEgZGlyZWN0ZWQgZ3JhcGggb2Ygb2JzZXJ2ZWQgc3RhdGVzIGFuZCBhY3Rpb24gdHJhbnNpdGlvbnMuCgpOb2RlcyBhcmUgc3RhdGUga2V5cyAoaGFzaCBieXRlcyBvZiB0aGUgbWFza2VkIGdyaWQpLiBFZGdlcyByZWNvcmQsIGZvciBlYWNoIChzdGF0ZSwKYWN0aW9uKSBhY3R1YWxseSB0YWtlbiwgdGhlIHJlc3VsdGluZyBzdGF0ZSBhbmQgdGhlIHJld2FyZCAoY2hhbmdlIGluIGxldmVsc19jb21wbGV0ZWQpLgpBc3N1bWVzIChsb2NhbGx5KSBkZXRlcm1pbmlzdGljIGR5bmFtaWNzOiBzdGF0ZSArIGFjdGlvbiAtPiBzYW1lIG5leHQgc3RhdGUuIFRoZSBncmFwaApkcml2ZXMgZXhwbG9yYXRpb24gKGZpbmQgbmVhcmVzdCB1bmV4cGxvcmVkIGZyb250aWVyKSBhbmQgZXhwbG9pdGF0aW9uIChyZXBsYXkgYWN0aW9uCnNlcXVlbmNlcyB0aGF0IGxlYWQgdG8gcmV3YXJkKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBIYXNoYWJsZSwgT3B0aW9uYWwKCiMgQW4gQWN0aW9uIGlzIGEgc21hbGwgaGFzaGFibGUgdG9rZW4gdGhlIGFnZW50IG1hcHMgdG8gYSBHYW1lQWN0aW9uOgojICAgKCJTIiwgYWN0aW9uX2lkKSAgICAgICAgICBzaW1wbGUgYWN0aW9uICgxLi41LCA3KQojICAgKCJDIiwgeCwgeSkgICAgICAgICAgICAgICBjb21wbGV4IGNsaWNrIChBQ1RJT042IGF0IHgseSkKQWN0aW9uID0gdHVwbGUKCgpAZGF0YWNsYXNzCmNsYXNzIE5vZGU6CiAgICBrZXk6IGJ5dGVzCiAgICBjYW5kaWRhdGVfYWN0aW9uczogdHVwbGVbQWN0aW9uLCAuLi5dID0gKCkgICMgZnVsbCBhY3Rpb24gc2V0IHByb3Bvc2VkIGF0IHRoaXMgc3RhdGUKICAgIGVkZ2VzOiBkaWN0W0FjdGlvbiwgdHVwbGVbYnl0ZXMsIGZsb2F0XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkgICMgYWN0aW9uIC0+IChuZXh0X2tleSwgcmV3YXJkKQogICAgdGVybWluYWw6IGJvb2wgPSBGYWxzZSAgIyBHQU1FX09WRVIgcmVhY2hlZCBoZXJlIChvbmx5IFJFU0VUIGVzY2FwZXMpCiAgICB2aXNpdHM6IGludCA9IDAKCiAgICBkZWYgdW50cmllZChzZWxmKSAtPiBsaXN0W0FjdGlvbl06CiAgICAgICAgcmV0dXJuIFthIGZvciBhIGluIHNlbGYuY2FuZGlkYXRlX2FjdGlvbnMgaWYgYSBub3QgaW4gc2VsZi5lZGdlc10KCgpjbGFzcyBXb3JsZE1vZGVsOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ub2RlczogZGljdFtieXRlcywgTm9kZV0gPSB7fQoKICAgIGRlZiBvYnNlcnZlKHNlbGYsIGtleTogYnl0ZXMsIGNhbmRpZGF0ZXM6IHR1cGxlW0FjdGlvbiwgLi4uXSwgdGVybWluYWw6IGJvb2wgPSBGYWxzZSkgLT4gTm9kZToKICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgIGlmIG5vZGUgaXMgTm9uZToKICAgICAgICAgICAgbm9kZSA9IE5vZGUoa2V5PWtleSwgY2FuZGlkYXRlX2FjdGlvbnM9Y2FuZGlkYXRlcywgdGVybWluYWw9dGVybWluYWwpCiAgICAgICAgICAgIHNlbGYubm9kZXNba2V5XSA9IG5vZGUKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIG1lcmdlIGFueSBuZXdseS1wcm9wb3NlZCBjYW5kaWRhdGVzIChrZWVwIG9yZGVyLCBkZWR1cCkKICAgICAgICAgICAgaWYgY2FuZGlkYXRlcyBhbmQgY2FuZGlkYXRlcyAhPSBub2RlLmNhbmRpZGF0ZV9hY3Rpb25zOgogICAgICAgICAgICAgICAgc2VlbiA9IHNldChub2RlLmNhbmRpZGF0ZV9hY3Rpb25zKQogICAgICAgICAgICAgICAgbWVyZ2VkID0gbGlzdChub2RlLmNhbmRpZGF0ZV9hY3Rpb25zKSArIFtjIGZvciBjIGluIGNhbmRpZGF0ZXMgaWYgYyBub3QgaW4gc2Vlbl0KICAgICAgICAgICAgICAgIG5vZGUuY2FuZGlkYXRlX2FjdGlvbnMgPSB0dXBsZShtZXJnZWQpCiAgICAgICAgICAgIG5vZGUudGVybWluYWwgPSBub2RlLnRlcm1pbmFsIG9yIHRlcm1pbmFsCiAgICAgICAgbm9kZS52aXNpdHMgKz0gMQogICAgICAgIHJldHVybiBub2RlCgogICAgZGVmIHJlY29yZChzZWxmLCBrZXk6IGJ5dGVzLCBhY3Rpb246IEFjdGlvbiwgbmV4dF9rZXk6IGJ5dGVzLCByZXdhcmQ6IGZsb2F0KSAtPiBOb25lOgogICAgICAgIG5vZGUgPSBzZWxmLm5vZGVzLmdldChrZXkpCiAgICAgICAgaWYgbm9kZSBpcyBOb25lOgogICAgICAgICAgICBub2RlID0gTm9kZShrZXk9a2V5KQogICAgICAgICAgICBzZWxmLm5vZGVzW2tleV0gPSBub2RlCiAgICAgICAgbm9kZS5lZGdlc1thY3Rpb25dID0gKG5leHRfa2V5LCByZXdhcmQpCgogICAgZGVmIGhhc191bnRyaWVkKHNlbGYsIGtleTogYnl0ZXMpIC0+IGJvb2w6CiAgICAgICAgbiA9IHNlbGYubm9kZXMuZ2V0KGtleSkKICAgICAgICByZXR1cm4gYm9vbChuIGFuZCBub3Qgbi50ZXJtaW5hbCBhbmQgbi51bnRyaWVkKCkpCgogICAgZGVmIHJld2FyZF9hY3Rpb24oc2VsZiwga2V5OiBieXRlcykgLT4gT3B0aW9uYWxbQWN0aW9uXToKICAgICAgICAiIiJJZiB0aGlzIHN0YXRlIGhhcyBhIGtub3duIGFjdGlvbiB0aGF0IHlpZWxkZWQgcG9zaXRpdmUgcmV3YXJkLCByZXR1cm4gaXQuIiIiCiAgICAgICAgbiA9IHNlbGYubm9kZXMuZ2V0KGtleSkKICAgICAgICBpZiBub3QgbjoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBiZXN0LCBiZXN0X3IgPSBOb25lLCAwLjAKICAgICAgICBmb3IgYSwgKF9uaywgcikgaW4gbi5lZGdlcy5pdGVtcygpOgogICAgICAgICAgICBpZiByID4gYmVzdF9yOgogICAgICAgICAgICAgICAgYmVzdCwgYmVzdF9yID0gYSwgcgogICAgICAgIHJldHVybiBiZXN0CgogICAgZGVmIHBhdGhfdG9fZnJvbnRpZXIoc2VsZiwgc3RhcnQ6IGJ5dGVzLCBtYXhfZGVwdGg6IGludCA9IDEwMDAwMCkgLT4gT3B0aW9uYWxbbGlzdFtBY3Rpb25dXToKICAgICAgICAiIiJCRlMgb3ZlciBrbm93biBlZGdlcyB0byB0aGUgbmVhcmVzdCBub24tdGVybWluYWwgbm9kZSB3aXRoIHVudHJpZWQgYWN0aW9ucy4KCiAgICAgICAgUmV0dXJucyB0aGUgYWN0aW9uIHNlcXVlbmNlIGZyb20gYHN0YXJ0YCB0byB0aGF0IGZyb250aWVyIG5vZGUgKGVtcHR5IGxpc3QgaWYKICAgICAgICBgc3RhcnRgIGl0c2VsZiBpcyBhIGZyb250aWVyKSwgb3IgTm9uZSBpZiBub25lIHJlYWNoYWJsZS4KICAgICAgICAiIiIKICAgICAgICBpZiBzZWxmLmhhc191bnRyaWVkKHN0YXJ0KToKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgdmlzaXRlZCA9IHtzdGFydH0KICAgICAgICAjIHF1ZXVlIG9mIChrZXksIHBhdGgpCiAgICAgICAgcTogZGVxdWVbdHVwbGVbYnl0ZXMsIGxpc3RbQWN0aW9uXV1dID0gZGVxdWUoWyhzdGFydCwgW10pXSkKICAgICAgICB3aGlsZSBxOgogICAgICAgICAgICBrZXksIHBhdGggPSBxLnBvcGxlZnQoKQogICAgICAgICAgICBpZiBsZW4ocGF0aCkgPiBtYXhfZGVwdGg6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgICAgICBpZiBub3Qgbm9kZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhY3Rpb24sIChuaywgX3IpIGluIG5vZGUuZWRnZXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIG5rIGluIHZpc2l0ZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHZpc2l0ZWQuYWRkKG5rKQogICAgICAgICAgICAgICAgbnBhdGggPSBwYXRoICsgW2FjdGlvbl0KICAgICAgICAgICAgICAgIGlmIHNlbGYuaGFzX3VudHJpZWQobmspOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBucGF0aAogICAgICAgICAgICAgICAgcS5hcHBlbmQoKG5rLCBucGF0aCkpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgcGF0aF9iZXR3ZWVuKHNlbGYsIHN0YXJ0OiBieXRlcywgZ29hbDogYnl0ZXMpIC0+IE9wdGlvbmFsW2xpc3RbQWN0aW9uXV06CiAgICAgICAgIiIiU2hvcnRlc3Qga25vd24gYWN0aW9uIHBhdGggZnJvbSBzdGFydCB0byBnb2FsLCBvciBOb25lLiIiIgogICAgICAgIGlmIHN0YXJ0ID09IGdvYWw6CiAgICAgICAgICAgIHJldHVybiBbXQogICAgICAgIHZpc2l0ZWQgPSB7c3RhcnR9CiAgICAgICAgcTogZGVxdWVbdHVwbGVbYnl0ZXMsIGxpc3RbQWN0aW9uXV1dID0gZGVxdWUoWyhzdGFydCwgW10pXSkKICAgICAgICB3aGlsZSBxOgogICAgICAgICAgICBrZXksIHBhdGggPSBxLnBvcGxlZnQoKQogICAgICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgICAgICBpZiBub3Qgbm9kZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhY3Rpb24sIChuaywgX3IpIGluIG5vZGUuZWRnZXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIG5rIGluIHZpc2l0ZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIG5rID09IGdvYWw6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIHBhdGggKyBbYWN0aW9uXQogICAgICAgICAgICAgICAgdmlzaXRlZC5hZGQobmspCiAgICAgICAgICAgICAgICBxLmFwcGVuZCgobmssIHBhdGggKyBbYWN0aW9uXSkpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLm5vZGVzKQo=', 'movement.py': 'IiIiTW90aW9uIG1vZGVsOiBsZWFybiB0aGUgY29udHJvbGxhYmxlIG9iamVjdCAoYXZhdGFyKSBhbmQgaG93IGFjdGlvbnMgbW92ZSBpdC4KCk1vc3QgQVJDLUFHSS0zIGdhbWVzIChhbmQgaW50ZXJhY3RpdmUgZ2FtZXMgZ2VuZXJhbGx5KSBoYXZlIGFuIGF2YXRhciB0aGUgcGxheWVyIG1vdmVzCndpdGggc2ltcGxlIGFjdGlvbnMuIElmIHdlIGNhbiBpZGVudGlmeSBpdCBhbmQgbGVhcm4gZWFjaCBhY3Rpb24ncyBkaXNwbGFjZW1lbnQgdmVjdG9yLAp3ZSBjYW4gbmF2aWdhdGUgaW4gY29vcmRpbmF0ZSBzcGFjZSAoY2hlYXAsIGdvYWwtZGlyZWN0ZWQpIGluc3RlYWQgb2YgYmxpbmQgZXhwbG9yYXRpb24Kb3ZlciBoYXNoZWQgc3RhdGVzLiBUaGlzIGlzIGZ1bGx5IGdlbmVyYWwg4oCUIG5vIHBlci1nYW1lIGtub3dsZWRnZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLiBpbXBvcnQgcGVyY2VwdGlvbiBhcyBQCgoKZGVmIGNvbG9yZWRfY2VsbHMoZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50KSAtPiBkaWN0W2ludCwgbnAubmRhcnJheV06CiAgICAiIiJNYXAgY29sb3IgLT4gYm9vbGVhbiBtYXNrIG9mIGl0cyBjZWxscyAoZXhjbHVkaW5nIGJhY2tncm91bmQpLiIiIgogICAgb3V0ID0ge30KICAgIGZvciBjIGluIG5wLnVuaXF1ZShncmlkKToKICAgICAgICBpZiBpbnQoYykgPT0gYmFja2dyb3VuZDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBvdXRbaW50KGMpXSA9IGdyaWQgPT0gYwogICAgcmV0dXJuIG91dAoKCmRlZiBpbmZlcl90cmFuc2xhdGlvbihiZWZvcmU6IG5wLm5kYXJyYXksIGFmdGVyOiBucC5uZGFycmF5LCBiYWNrZ3JvdW5kOiBpbnQpOgogICAgIiIiSWYgZXhhY3RseSBvbmUgY29sb3IncyByZWdpb24gdHJhbnNsYXRlZCBieSBhIGNvbnN0YW50IHZlY3RvciwgcmV0dXJuIChjb2xvciwgZHIsIGRjKS4KCiAgICBSZXR1cm5zIE5vbmUgaWYgdGhlIGNoYW5nZSBpc24ndCBhIGNsZWFuIHNpbmdsZS1vYmplY3QgdHJhbnNsYXRpb24uCiAgICAiIiIKICAgIGlmIGJlZm9yZS5zaGFwZSAhPSBhZnRlci5zaGFwZToKICAgICAgICByZXR1cm4gTm9uZQogICAgY2hhbmdlZF9jb2xvcnMgPSBbXQogICAgZm9yIGMgaW4gc2V0KG5wLnVuaXF1ZShiZWZvcmUpKS51bmlvbihucC51bmlxdWUoYWZ0ZXIpKToKICAgICAgICBjID0gaW50KGMpCiAgICAgICAgaWYgYyA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGIgPSBiZWZvcmUgPT0gYwogICAgICAgIGEgPSBhZnRlciA9PSBjCiAgICAgICAgaWYgbm90IG5wLmFycmF5X2VxdWFsKGIsIGEpOgogICAgICAgICAgICBjaGFuZ2VkX2NvbG9ycy5hcHBlbmQoYykKICAgICMgVGhlIGF2YXRhciBpcyBhIGNvbG9yIHdob3NlIG1hc2sgbW92ZWQuIFN0YXRpYyBkZWNvcmF0aW9ucyBkb24ndCBjaGFuZ2UuCiAgICBiZXN0ID0gTm9uZQogICAgZm9yIGMgaW4gY2hhbmdlZF9jb2xvcnM6CiAgICAgICAgYiA9IG5wLmFyZ3doZXJlKGJlZm9yZSA9PSBjKQogICAgICAgIGEgPSBucC5hcmd3aGVyZShhZnRlciA9PSBjKQogICAgICAgIGlmIGxlbihiKSA9PSAwIG9yIGxlbihhKSA9PSAwIG9yIGxlbihiKSAhPSBsZW4oYSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBjYW5kaWRhdGUgdHJhbnNsYXRpb24gPSBjZW50cm9pZCBzaGlmdAogICAgICAgIGRiID0gYi5tZWFuKGF4aXM9MCkKICAgICAgICBkYSA9IGEubWVhbihheGlzPTApCiAgICAgICAgZHIsIGRjID0gZGEgLSBkYiwgTm9uZQogICAgICAgIHNoaWZ0ID0gKGRhIC0gZGIpCiAgICAgICAgIyB2ZXJpZnkgaXQncyBhIHJpZ2lkIHRyYW5zbGF0aW9uOiBzaGlmdGluZyBiZWZvcmUtY2VsbHMgYnkgcm91bmQoc2hpZnQpID09IGFmdGVyLWNlbGxzCiAgICAgICAgc3IsIHNjID0gaW50KHJvdW5kKHNoaWZ0WzBdKSksIGludChyb3VuZChzaGlmdFsxXSkpCiAgICAgICAgc2hpZnRlZCA9IGIgKyBucC5hcnJheShbc3IsIHNjXSkKICAgICAgICBpZiBzZXQobWFwKHR1cGxlLCBzaGlmdGVkLnRvbGlzdCgpKSkgPT0gc2V0KG1hcCh0dXBsZSwgYS50b2xpc3QoKSkpOgogICAgICAgICAgICBpZiAoc3IsIHNjKSAhPSAoMCwgMCk6CiAgICAgICAgICAgICAgICAjIHByZWZlciB0aGUgc21hbGxlc3QgbW92aW5nIG9iamVjdCAobGlrZWx5IHRoZSBhdmF0YXIpCiAgICAgICAgICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3IgbGVuKGIpIDwgYmVzdFszXToKICAgICAgICAgICAgICAgICAgICBiZXN0ID0gKGMsIHNyLCBzYywgbGVuKGIpKQogICAgaWYgYmVzdCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gKGJlc3RbMF0sIGJlc3RbMV0sIGJlc3RbMl0pCgoKZGVmIGluZmVyX2FsbF90cmFuc2xhdGlvbnMoYmVmb3JlOiBucC5uZGFycmF5LCBhZnRlcjogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50KSAtPiBkaWN0OgogICAgIiIiUmV0dXJuIHtjb2xvcjogKGRyLCBkYyl9IGZvciBldmVyeSBub24tYmFja2dyb3VuZCBjb2xvciB0aGF0IHJpZ2lkbHkgdHJhbnNsYXRlZC4KCiAgICBVbmxpa2UgaW5mZXJfdHJhbnNsYXRpb24gKHNpbmdsZSBiZXN0IG1vdmVyKSwgdGhpcyByZXBvcnRzIGFsbCBtb3ZlcnMgc28gdGhlIGNhbGxlcgogICAgY2FuIGRpc3Rpbmd1aXNoIHRoZSBhdmF0YXIgKG1vdGlvbiB2YXJpZXMgd2l0aCB0aGUgYWN0aW9uKSBmcm9tIGluZGVwZW5kZW50CiAgICBhbmltYXRpb25zL2NvdW50ZXJzIChtb3Rpb24gaXMgY29uc3RhbnQgcmVnYXJkbGVzcyBvZiB0aGUgYWN0aW9uKS4KICAgICIiIgogICAgb3V0OiBkaWN0W2ludCwgdHVwbGVbaW50LCBpbnRdXSA9IHt9CiAgICBpZiBiZWZvcmUuc2hhcGUgIT0gYWZ0ZXIuc2hhcGU6CiAgICAgICAgcmV0dXJuIG91dAogICAgZm9yIGMgaW4gc2V0KG5wLnVuaXF1ZShiZWZvcmUpKS51bmlvbihucC51bmlxdWUoYWZ0ZXIpKToKICAgICAgICBjID0gaW50KGMpCiAgICAgICAgaWYgYyA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGIgPSBucC5hcmd3aGVyZShiZWZvcmUgPT0gYykKICAgICAgICBhID0gbnAuYXJnd2hlcmUoYWZ0ZXIgPT0gYykKICAgICAgICBpZiBsZW4oYikgPT0gMCBvciBsZW4oYSkgPT0gMCBvciBsZW4oYikgIT0gbGVuKGEpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNoaWZ0ID0gYS5tZWFuKGF4aXM9MCkgLSBiLm1lYW4oYXhpcz0wKQogICAgICAgIHNyLCBzYyA9IGludChyb3VuZChzaGlmdFswXSkpLCBpbnQocm91bmQoc2hpZnRbMV0pKQogICAgICAgIGlmIChzciwgc2MpID09ICgwLCAwKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzaGlmdGVkID0gYiArIG5wLmFycmF5KFtzciwgc2NdKQogICAgICAgIGlmIHNldChtYXAodHVwbGUsIHNoaWZ0ZWQudG9saXN0KCkpKSA9PSBzZXQobWFwKHR1cGxlLCBhLnRvbGlzdCgpKSk6CiAgICAgICAgICAgIG91dFtjXSA9IChzciwgc2MpCiAgICByZXR1cm4gb3V0CgoKQGRhdGFjbGFzcwpjbGFzcyBNb3Rpb25Nb2RlbDoKICAgIGF2YXRhcl9jb2xvcjogaW50IHwgTm9uZSA9IE5vbmUgICMgcHJpbWFyeSBjb2xvciAoZm9yIGRlbHRhIGxvb2t1cCkKICAgIGRlbHRhczogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkgICMgYWN0aW9uX2lkIC0+IChkcixkYykKICAgIGF2YXRhcl9jb2xvcnM6IGZyb3plbnNldCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1mcm96ZW5zZXQpICAjIGFsbCBjb2xvcnMgbW92aW5nIGFzIHRoZSBhdmF0YXIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBvayhzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLmF2YXRhcl9jb2xvciBpcyBub3QgTm9uZSBhbmQgbGVuKHNlbGYuZGVsdGFzKSA+IDAKCiAgICBkZWYgX21hc2soc2VsZiwgZ3JpZDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICBjb2xzID0gc2VsZi5hdmF0YXJfY29sb3JzIG9yICh7c2VsZi5hdmF0YXJfY29sb3J9IGlmIHNlbGYuYXZhdGFyX2NvbG9yIGlzIG5vdCBOb25lIGVsc2Ugc2V0KCkpCiAgICAgICAgcmV0dXJuIG5wLmlzaW4oZ3JpZCwgbGlzdChjb2xzKSkKCiAgICBkZWYgYXZhdGFyX2NlbnRyb2lkKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpOgogICAgICAgICIiIkNlbnRyb2lkIChyb3csY29sKSBvdmVyIEFMTCBhdmF0YXIgY29sb3JzIChtdWx0aS1jb2xvciBhdmF0YXJzIG1vdmUgdG9nZXRoZXIpLiIiIgogICAgICAgIGNlbGxzID0gbnAuYXJnd2hlcmUoc2VsZi5fbWFzayhncmlkKSkKICAgICAgICBpZiBsZW4oY2VsbHMpID09IDA6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0dXJuIHR1cGxlKGNlbGxzLm1lYW4oYXhpcz0wKSkKCiAgICBkZWYgYXZhdGFyX2NlbGxzKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIG5wLmFyZ3doZXJlKHNlbGYuX21hc2soZ3JpZCkpCg==', 'agent.py': 'IiIiQWdlbnRzIGZvciBBUkMtQUdJLTMuCgotIEdyYXBoU3RyYXRlZ3k6IGdyYXBoLWJhc2VkIGV4cGxvcmF0aW9uL2V4cGxvaXRhdGlvbiBvdmVyIGEgV29ybGRNb2RlbCAoZnJvbnRpZXIgc2VhcmNoCiAgKyBzaG9ydGVzdC1wYXRoIHJlcGxheSArIHJld2FyZCBleHBsb2l0YXRpb24pLiBSZXVzYWJsZSBkZWNpc2lvbiBwb2xpY3kuCi0gRXhwbG9yZXJBZ2VudDogcHVyZSBncmFwaCBleHBsb3JlciAoYmFzZWxpbmUpLgotIEh5YnJpZEFnZW50OiBsZWFybnMgYSBtb3Rpb24gbW9kZWwgKGNvbnRyb2xsYWJsZSBhdmF0YXIgKyBwZXItYWN0aW9uIGRpc3BsYWNlbWVudCkgYW5kCiAgbmF2aWdhdGVzIGluIGNvb3JkaW5hdGUgc3BhY2UgdG8gY2FuZGlkYXRlIGdvYWwgb2JqZWN0czsgZmFsbHMgYmFjayB0byBHcmFwaFN0cmF0ZWd5CiAgd2hlbiBubyBhdmF0YXIgaXMgZm91bmQgb3IgbW90aW9uIHByb2dyZXNzIHN0YWxscy4KCkFsbCB0cmFpbmluZy1mcmVlIGFuZCBnYW1lLWFnbm9zdGljLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBsb2dnaW5nCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gYXJjZW5naW5lIGltcG9ydCBHYW1lQWN0aW9uLCBHYW1lU3RhdGUKCmZyb20gLiBpbXBvcnQgbW92ZW1lbnQgYXMgTVYKZnJvbSAuIGltcG9ydCBwZXJjZXB0aW9uIGFzIFAKZnJvbSAud29ybGRfbW9kZWwgaW1wb3J0IEFjdGlvbiwgV29ybGRNb2RlbAoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyY2FnaTMuYWdlbnQiKQoKU0lNUExFX0lEUyA9IFsxLCAyLCAzLCA0LCA1LCA3XSAgIyBSRVNFVCgwKS9BQ1RJT042KGNsaWNrKSBoYW5kbGVkIHNlcGFyYXRlbHkKCgpAZGF0YWNsYXNzCmNsYXNzIFBsYXlSZXN1bHQ6CiAgICBnYW1lX2lkOiBzdHIKICAgIGxldmVsc19jb21wbGV0ZWQ6IGludAogICAgd2luX2xldmVsczogaW50CiAgICBhY3Rpb25zOiBpbnQKICAgIHdvbjogYm9vbAogICAgc3RhdGVzX3NlZW46IGludAogICAgcmVhc29uOiBzdHIgPSAiIgoKCmRlZiB0b19nYW1lX2FjdGlvbihhOiBBY3Rpb24pIC0+IHR1cGxlW0dhbWVBY3Rpb24sIGRpY3RdOgogICAgaWYgYVswXSA9PSAiUyI6CiAgICAgICAgcmV0dXJuIEdhbWVBY3Rpb24uZnJvbV9pZChhWzFdKSwge30KICAgIGlmIGFbMF0gPT0gIkMiOgogICAgICAgIHJldHVybiBHYW1lQWN0aW9uLkFDVElPTjYsIHsieCI6IGFbMV0sICJ5IjogYVsyXX0KICAgIHJhaXNlIFZhbHVlRXJyb3IoYSkKCgpkZWYgY2FuZGlkYXRlc19mb3IoZ3JpZDogbnAubmRhcnJheSwgYXZhaWxhYmxlOiBsaXN0W2ludF0sIHVzZV9jbGlja3M6IGJvb2wsCiAgICAgICAgICAgICAgICAgICBtYXhfY2xpY2tfdGFyZ2V0czogaW50LCB1c2VfdW5kbzogYm9vbCkgLT4gdHVwbGVbQWN0aW9uLCAuLi5dOgogICAgY2FuZHM6IGxpc3RbQWN0aW9uXSA9IFtdCiAgICBmb3IgYWlkIGluIFNJTVBMRV9JRFM6CiAgICAgICAgaWYgYWlkIGluIGF2YWlsYWJsZSBhbmQgKGFpZCAhPSA3IG9yIHVzZV91bmRvKToKICAgICAgICAgICAgY2FuZHMuYXBwZW5kKCgiUyIsIGFpZCkpCiAgICBpZiB1c2VfY2xpY2tzIGFuZCA2IGluIGF2YWlsYWJsZToKICAgICAgICBmb3IgeCwgeSwgX3ByaW8gaW4gUC5zYWxpZW50X2NsaWNrX3RhcmdldHMoCiAgICAgICAgICAgIGdyaWQsIG1heF90YXJnZXRzPW1heF9jbGlja190YXJnZXRzLCBjb2Fyc2VfZ3JpZF9zdGVwPTgKICAgICAgICApOgogICAgICAgICAgICBjYW5kcy5hcHBlbmQoKCJDIiwgaW50KHgpLCBpbnQoeSkpKQogICAgcmV0dXJuIHR1cGxlKGNhbmRzKQoKCiMgLS0tIGRlY2lzaW9ucyB0aGUgc3RyYXRlZ3kgY2FuIHJldHVybiB0byB0aGUgZHJpdmluZyBsb29wIC0tLQpBQ1QgPSAiYWN0IgpSRVNFVCA9ICJyZXNldCIKU1RPUCA9ICJzdG9wIgoKCmNsYXNzIEdyYXBoU3RyYXRlZ3k6CiAgICAiIiJTdGF0ZWZ1bCBncmFwaCBleHBsb3JhdGlvbiBwb2xpY3kgb3ZlciBhIHNoYXJlZCBXb3JsZE1vZGVsLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb290X2tleTogYnl0ZXMsIG1heF9zdHVja19yZXNldHM6IGludCA9IDUwKSAtPiBOb25lOgogICAgICAgIHNlbGYud20gPSBXb3JsZE1vZGVsKCkKICAgICAgICBzZWxmLnJvb3Rfa2V5ID0gcm9vdF9rZXkKICAgICAgICBzZWxmLnBsYW46IGxpc3RbQWN0aW9uXSA9IFtdCiAgICAgICAgc2VsZi5fZXhwZWN0OiBieXRlcyB8IE5vbmUgPSBOb25lCiAgICAgICAgc2VsZi5zdHVja19yZXNldHMgPSAwCiAgICAgICAgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzID0gbWF4X3N0dWNrX3Jlc2V0cwoKICAgIGRlZiBkZWNpZGUoc2VsZiwgY3VyX2tleTogYnl0ZXMpIC0+IHR1cGxlW3N0ciwgQWN0aW9uIHwgTm9uZV06CiAgICAgICAgbm9kZSA9IHNlbGYud20ubm9kZXMuZ2V0KGN1cl9rZXkpCiAgICAgICAgaWYgbm9kZSBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCgogICAgICAgICMgMSkgZXhwbG9pdCBhIGtub3duIHJld2FyZC1wcm9kdWNpbmcgYWN0aW9uCiAgICAgICAgcl9hY3QgPSBzZWxmLndtLnJld2FyZF9hY3Rpb24oY3VyX2tleSkKICAgICAgICBpZiByX2FjdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIChBQ1QsIHJfYWN0KQoKICAgICAgICAjIDIpIGNvbnRpbnVlIGFuIGFjdGl2ZSBwbGFuIChyZXBsYXkpLCBhYm9ydGluZyBvbiBkaXZlcmdlbmNlCiAgICAgICAgaWYgc2VsZi5wbGFuOgogICAgICAgICAgICBpZiBzZWxmLl9leHBlY3QgaXMgbm90IE5vbmUgYW5kIGN1cl9rZXkgIT0gc2VsZi5fZXhwZWN0OgogICAgICAgICAgICAgICAgc2VsZi5wbGFuID0gW10KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJldHVybiAoQUNULCBzZWxmLnBsYW4ucG9wKDApKQoKICAgICAgICAjIDMpIHVudHJpZWQgY2FuZGlkYXRlIGhlcmUKICAgICAgICBpZiBub2RlLnVudHJpZWQoKToKICAgICAgICAgICAgcmV0dXJuIChBQ1QsIG5vZGUudW50cmllZCgpWzBdKQoKICAgICAgICAjIDQpIG5hdmlnYXRlIHRvIG5lYXJlc3QgZnJvbnRpZXIKICAgICAgICBwYXRoID0gc2VsZi53bS5wYXRoX3RvX2Zyb250aWVyKGN1cl9rZXkpCiAgICAgICAgaWYgcGF0aCBpcyBOb25lOgogICAgICAgICAgICBpZiBjdXJfa2V5ICE9IHNlbGYucm9vdF9rZXkgYW5kIHNlbGYuc3R1Y2tfcmVzZXRzIDwgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzOgogICAgICAgICAgICAgICAgc2VsZi5zdHVja19yZXNldHMgKz0gMQogICAgICAgICAgICAgICAgcmV0dXJuIChSRVNFVCwgTm9uZSkKICAgICAgICAgICAgcGF0aCA9IHNlbGYud20ucGF0aF90b19mcm9udGllcihzZWxmLnJvb3Rfa2V5KQogICAgICAgICAgICBpZiBwYXRoIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCiAgICAgICAgaWYgcGF0aDoKICAgICAgICAgICAgc2VsZi5wbGFuID0gcGF0aAogICAgICAgICAgICByZXR1cm4gKEFDVCwgc2VsZi5wbGFuLnBvcCgwKSkKICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCgogICAgZGVmIHVwZGF0ZShzZWxmLCBjdXJfa2V5OiBieXRlcywgYWN0aW9uOiBBY3Rpb24sIG5leHRfa2V5OiBieXRlcywgcmV3YXJkOiBmbG9hdCwKICAgICAgICAgICAgICAgY2FuZGlkYXRlczogdHVwbGVbQWN0aW9uLCAuLi5dLCB0ZXJtaW5hbDogYm9vbCkgLT4gTm9uZToKICAgICAgICBzZWxmLndtLnJlY29yZChjdXJfa2V5LCBhY3Rpb24sIG5leHRfa2V5LCByZXdhcmQpCiAgICAgICAgc2VsZi53bS5vYnNlcnZlKG5leHRfa2V5LCBjYW5kaWRhdGVzLCB0ZXJtaW5hbD10ZXJtaW5hbCkKICAgICAgICBzZWxmLl9leHBlY3QgPSBuZXh0X2tleSBpZiBzZWxmLnBsYW4gZWxzZSBOb25lCgoKY2xhc3MgRXhwbG9yZXJBZ2VudDoKICAgICIiIlB1cmUgZ3JhcGgtYmFzZWQgZXhwbG9yZXIgKGJhc2VsaW5lKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbWF4X2FjdGlvbnM6IGludCA9IDQwMDAsIHVzZV9jbGlja3M6IGJvb2wgPSBUcnVlLAogICAgICAgICAgICAgICAgIG1heF9jbGlja190YXJnZXRzOiBpbnQgPSA5NiwgdXNlX3VuZG86IGJvb2wgPSBGYWxzZSwgc2VlZDogaW50ID0gMCkgLT4gTm9uZToKICAgICAgICBzZWxmLm1heF9hY3Rpb25zID0gbWF4X2FjdGlvbnMKICAgICAgICBzZWxmLnVzZV9jbGlja3MgPSB1c2VfY2xpY2tzCiAgICAgICAgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cyA9IG1heF9jbGlja190YXJnZXRzCiAgICAgICAgc2VsZi51c2VfdW5kbyA9IHVzZV91bmRvCgogICAgZGVmIF9jYW5kcyhzZWxmLCBncmlkLCBhdmFpbGFibGUpOgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzX2ZvcihncmlkLCBhdmFpbGFibGUsIHNlbGYudXNlX2NsaWNrcywgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cywgc2VsZi51c2VfdW5kbykKCiAgICBkZWYgcGxheShzZWxmLCBlbnYsIGdhbWVfaWQ6IHN0ciA9ICI/IikgLT4gUGxheVJlc3VsdDoKICAgICAgICB2dCA9IFAuVm9sYXRpbGl0eVRyYWNrZXIoKQogICAgICAgIG9icyA9IGVudi5yZXNldCgpCiAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpCiAgICAgICAgdnQudXBkYXRlKGdyaWQpCiAgICAgICAgcm9vdF9rZXkgPSBQLnN0YXRlX2hhc2goZ3JpZCwgdnQubWFzaygpKQogICAgICAgIGdzID0gR3JhcGhTdHJhdGVneShyb290X2tleSkKICAgICAgICBncy53bS5vYnNlcnZlKHJvb3Rfa2V5LCBzZWxmLl9jYW5kcyhncmlkLCBvYnMuYXZhaWxhYmxlX2FjdGlvbnMpKQogICAgICAgIGN1cl9rZXkgPSByb290X2tleQogICAgICAgIGFjdGlvbnMgPSAwCiAgICAgICAgcHJldl9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICB3aW5fbGV2ZWxzID0gaW50KG9icy53aW5fbGV2ZWxzIG9yIDApCiAgICAgICAgcmVhc29uID0gImJ1ZGdldCIKCiAgICAgICAgd2hpbGUgYWN0aW9ucyA8IHNlbGYubWF4X2FjdGlvbnM6CiAgICAgICAgICAgIGlmIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuV0lOOgogICAgICAgICAgICAgICAgcmVhc29uID0gIndpbiIKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuR0FNRV9PVkVSOgogICAgICAgICAgICAgICAgbiA9IGdzLndtLm5vZGVzLmdldChjdXJfa2V5KQogICAgICAgICAgICAgICAgaWYgbjoKICAgICAgICAgICAgICAgICAgICBuLnRlcm1pbmFsID0gVHJ1ZQogICAgICAgICAgICAgICAgb2JzID0gZW52LnJlc2V0KCk7IGFjdGlvbnMgKz0gMQogICAgICAgICAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpOyB2dC51cGRhdGUoZ3JpZCk7IGN1cl9rZXkgPSByb290X2tleTsgZ3MucGxhbiA9IFtdCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAga2luZCwgYWN0aW9uID0gZ3MuZGVjaWRlKGN1cl9rZXkpCiAgICAgICAgICAgIGlmIGtpbmQgPT0gU1RPUDoKICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJleGhhdXN0ZWQiOyBicmVhawogICAgICAgICAgICBpZiBraW5kID09IFJFU0VUOgogICAgICAgICAgICAgICAgb2JzID0gZW52LnJlc2V0KCk7IGFjdGlvbnMgKz0gMQogICAgICAgICAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpOyB2dC51cGRhdGUoZ3JpZCk7IGN1cl9rZXkgPSByb290X2tleTsgZ3MucGxhbiA9IFtdCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgb2JzLCBncmlkLCBjdXJfa2V5LCBwcmV2X2xldmVscyA9IHNlbGYuX3N0ZXAoZW52LCBhY3Rpb24sIGdzLCB2dCwgY3VyX2tleSwgcHJldl9sZXZlbHMpCiAgICAgICAgICAgIGFjdGlvbnMgKz0gMQoKICAgICAgICByZXR1cm4gUGxheVJlc3VsdChnYW1lX2lkLCBwcmV2X2xldmVscywgd2luX2xldmVscywgYWN0aW9ucywKICAgICAgICAgICAgICAgICAgICAgICAgICBvYnMuc3RhdGUgPT0gR2FtZVN0YXRlLldJTiwgbGVuKGdzLndtKSwgcmVhc29uKQoKICAgIGRlZiBfc3RlcChzZWxmLCBlbnYsIGFjdGlvbiwgZ3MsIHZ0LCBjdXJfa2V5LCBwcmV2X2xldmVscyk6CiAgICAgICAgZ2EsIGRhdGEgPSB0b19nYW1lX2FjdGlvbihhY3Rpb24pCiAgICAgICAgb2JzID0gZW52LnN0ZXAoZ2EsIGRhdGE9ZGF0YSkgaWYgZGF0YSBlbHNlIGVudi5zdGVwKGdhKQogICAgICAgIG5ncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShuZ3JpZCkKICAgICAgICBua2V5ID0gUC5zdGF0ZV9oYXNoKG5ncmlkLCB2dC5tYXNrKCkpCiAgICAgICAgbmxldmVscyA9IGludChvYnMubGV2ZWxzX2NvbXBsZXRlZCBvciAwKQogICAgICAgIHRlcm1pbmFsID0gb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5HQU1FX09WRVIKICAgICAgICBncy51cGRhdGUoY3VyX2tleSwgYWN0aW9uLCBua2V5LCBmbG9hdChubGV2ZWxzIC0gcHJldl9sZXZlbHMpLAogICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kcyhuZ3JpZCwgb2JzLmF2YWlsYWJsZV9hY3Rpb25zKSwgdGVybWluYWwpCiAgICAgICAgcmV0dXJuIG9icywgbmdyaWQsIG5rZXksIG5sZXZlbHMKCgpjbGFzcyBIeWJyaWRBZ2VudDoKICAgICIiIk1vdGlvbi1maXJzdCBhZ2VudDogbGVhcm4gdGhlIGF2YXRhciArIHBlci1hY3Rpb24gZGlzcGxhY2VtZW50LCBuYXZpZ2F0ZSB0byBnb2FsCiAgICBvYmplY3RzIGluIGNvb3JkaW5hdGUgc3BhY2U7IGZhbGwgYmFjayB0byBncmFwaCBleHBsb3JhdGlvbiB3aGVuIHN0YWxsZWQuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1heF9hY3Rpb25zOiBpbnQgPSA0MDAwLCB1c2VfY2xpY2tzOiBib29sID0gVHJ1ZSwKICAgICAgICAgICAgICAgICBtYXhfY2xpY2tfdGFyZ2V0czogaW50ID0gOTYsIG5hdl9zdGVwX2NhcDogaW50ID0gMjAwLCBzZWVkOiBpbnQgPSAwKSAtPiBOb25lOgogICAgICAgIHNlbGYubWF4X2FjdGlvbnMgPSBtYXhfYWN0aW9ucwogICAgICAgIHNlbGYudXNlX2NsaWNrcyA9IHVzZV9jbGlja3MKICAgICAgICBzZWxmLm1heF9jbGlja190YXJnZXRzID0gbWF4X2NsaWNrX3RhcmdldHMKICAgICAgICBzZWxmLm5hdl9zdGVwX2NhcCA9IG5hdl9zdGVwX2NhcAogICAgICAgIHNlbGYucm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCgogICAgZGVmIF9jYW5kcyhzZWxmLCBncmlkLCBhdmFpbGFibGUpOgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzX2ZvcihncmlkLCBhdmFpbGFibGUsIHNlbGYudXNlX2NsaWNrcywgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cywgRmFsc2UpCgogICAgZGVmIHBsYXkoc2VsZiwgZW52LCBnYW1lX2lkOiBzdHIgPSAiPyIpIC0+IFBsYXlSZXN1bHQ6CiAgICAgICAgdnQgPSBQLlZvbGF0aWxpdHlUcmFja2VyKCkKICAgICAgICBvYnMgPSBlbnYucmVzZXQoKQogICAgICAgIGdyaWQgPSBQLnRvX2dyaWQob2JzLmZyYW1lKTsgdnQudXBkYXRlKGdyaWQpCiAgICAgICAgcm9vdF9rZXkgPSBQLnN0YXRlX2hhc2goZ3JpZCwgdnQubWFzaygpKQogICAgICAgIGdzID0gR3JhcGhTdHJhdGVneShyb290X2tleSkKICAgICAgICBncy53bS5vYnNlcnZlKHJvb3Rfa2V5LCBzZWxmLl9jYW5kcyhncmlkLCBvYnMuYXZhaWxhYmxlX2FjdGlvbnMpKQogICAgICAgIGN1cl9rZXkgPSByb290X2tleQogICAgICAgIGFjdGlvbnMgPSAwCiAgICAgICAgcHJldl9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICB3aW5fbGV2ZWxzID0gaW50KG9icy53aW5fbGV2ZWxzIG9yIDApCiAgICAgICAgcmVhc29uID0gImJ1ZGdldCIKCiAgICAgICAgYmcgPSBQLmRldGVjdF9iYWNrZ3JvdW5kKGdyaWQpCiAgICAgICAgbW06IE1WLk1vdGlvbk1vZGVsIHwgTm9uZSA9IE5vbmUKICAgICAgICBsZXZlbF9vZl9tb2RlbCA9IC0xCiAgICAgICAgdHJpZWRfdGFyZ2V0czogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgICAgIG1vdGlvbl9kZWFkID0gRmFsc2UgICMgYXZhdGFyIHN0cmF0ZWd5IGdhdmUgdXAgZm9yIHRoaXMgbGV2ZWwKCiAgICAgICAgZGVmIHJlY29yZChhY3Rpb24sIG9ic19uZXcpOgogICAgICAgICAgICBub25sb2NhbCBjdXJfa2V5LCBwcmV2X2xldmVscywgZ3JpZCwgYWN0aW9ucwogICAgICAgICAgICBuZ3JpZCA9IFAudG9fZ3JpZChvYnNfbmV3LmZyYW1lKTsgdnQudXBkYXRlKG5ncmlkKQogICAgICAgICAgICBua2V5ID0gUC5zdGF0ZV9oYXNoKG5ncmlkLCB2dC5tYXNrKCkpCiAgICAgICAgICAgIG5sZXZlbHMgPSBpbnQob2JzX25ldy5sZXZlbHNfY29tcGxldGVkIG9yIDApCiAgICAgICAgICAgIHRlcm1pbmFsID0gb2JzX25ldy5zdGF0ZSA9PSBHYW1lU3RhdGUuR0FNRV9PVkVSCiAgICAgICAgICAgIGdzLnVwZGF0ZShjdXJfa2V5LCBhY3Rpb24sIG5rZXksIGZsb2F0KG5sZXZlbHMgLSBwcmV2X2xldmVscyksCiAgICAgICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kcyhuZ3JpZCwgb2JzX25ldy5hdmFpbGFibGVfYWN0aW9ucyksIHRlcm1pbmFsKQogICAgICAgICAgICBjdXJfa2V5LCBwcmV2X2xldmVscywgZ3JpZCA9IG5rZXksIG5sZXZlbHMsIG5ncmlkCiAgICAgICAgICAgIGFjdGlvbnMgKz0gMQoKICAgICAgICB3aGlsZSBhY3Rpb25zIDwgc2VsZi5tYXhfYWN0aW9uczoKICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICByZWFzb24gPSAid2luIjsgYnJlYWsKICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5HQU1FX09WRVI6CiAgICAgICAgICAgICAgICBuID0gZ3Mud20ubm9kZXMuZ2V0KGN1cl9rZXkpCiAgICAgICAgICAgICAgICBpZiBuOgogICAgICAgICAgICAgICAgICAgIG4udGVybWluYWwgPSBUcnVlCiAgICAgICAgICAgICAgICBvYnMgPSBlbnYucmVzZXQoKTsgYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShncmlkKTsgY3VyX2tleSA9IHJvb3Rfa2V5OyBncy5wbGFuID0gW10KICAgICAgICAgICAgICAgIG1tID0gTm9uZTsgbW90aW9uX2RlYWQgPSBGYWxzZTsgdHJpZWRfdGFyZ2V0cy5jbGVhcigpCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyBOZXcgbGV2ZWwgLT4gcmVsZWFybiBtb3Rpb24KICAgICAgICAgICAgaWYgcHJldl9sZXZlbHMgIT0gbGV2ZWxfb2ZfbW9kZWw6CiAgICAgICAgICAgICAgICBtbSA9IE5vbmU7IG1vdGlvbl9kZWFkID0gRmFsc2U7IHRyaWVkX3RhcmdldHMuY2xlYXIoKQogICAgICAgICAgICAgICAgbGV2ZWxfb2ZfbW9kZWwgPSBwcmV2X2xldmVscwoKICAgICAgICAgICAgc2ltcGxlX2F2YWlsID0gW2EgZm9yIGEgaW4gKDEsIDIsIDMsIDQsIDUpIGlmIGEgaW4gb2JzLmF2YWlsYWJsZV9hY3Rpb25zXQoKICAgICAgICAgICAgIyAtLS0tIGxlYXJuIG1vdGlvbiBtb2RlbCBieSBwcm9iaW5nIHNpbXBsZSBhY3Rpb25zIC0tLS0KICAgICAgICAgICAgaWYgbW0gaXMgTm9uZSBhbmQgc2ltcGxlX2F2YWlsIGFuZCBub3QgbW90aW9uX2RlYWQ6CiAgICAgICAgICAgICAgICBtbSA9IHNlbGYuX2xlYXJuX21vdGlvbihlbnYsIG9icywgZ3JpZCwgYmcsIHNpbXBsZV9hdmFpbCwgcmVjb3JkX2ZuPXJlY29yZCkKICAgICAgICAgICAgICAgIG9icyA9IHNlbGYuX2xhc3Rfb2JzCiAgICAgICAgICAgICAgICBpZiBtbSBpcyBOb25lIG9yIG5vdCBtbS5kZWx0YXM6CiAgICAgICAgICAgICAgICAgICAgbW90aW9uX2RlYWQgPSBUcnVlCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyAtLS0tIG5hdmlnYXRlIGF2YXRhciB0byBhIGNhbmRpZGF0ZSBnb2FsIG9iamVjdCAtLS0tCiAgICAgICAgICAgIGlmIG1tIGlzIG5vdCBOb25lIGFuZCBtbS5vayBhbmQgbm90IG1vdGlvbl9kZWFkOgogICAgICAgICAgICAgICAgdGFyZ2V0ID0gc2VsZi5fbmV4dF90YXJnZXQoZ3JpZCwgYmcsIG1tLCB0cmllZF90YXJnZXRzKQogICAgICAgICAgICAgICAgaWYgdGFyZ2V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgbW90aW9uX2RlYWQgPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHRyaWVkX3RhcmdldHMuYWRkKHRhcmdldCkKICAgICAgICAgICAgICAgIG9icyA9IHNlbGYuX25hdmlnYXRlKGVudiwgbW0sIHRhcmdldCwgcmVjb3JkX2ZuPXJlY29yZCwgc3RhcnRfbGV2ZWxzPXByZXZfbGV2ZWxzKQogICAgICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICAgICAgcmVhc29uID0gIndpbiI7IGJyZWFrCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyAtLS0tIGdyYXBoIGZhbGxiYWNrIC0tLS0KICAgICAgICAgICAga2luZCwgYWN0aW9uID0gZ3MuZGVjaWRlKGN1cl9rZXkpCiAgICAgICAgICAgIGlmIGtpbmQgPT0gU1RPUDoKICAgICAgICAgICAgICAgICMgbGFzdCByZXNvcnQ6IGlmIG1vdGlvbiBleGlzdGVkLCByZXNldCBhbmQgbGV0IG1vdGlvbiByZXRyeSBmcmVzaAogICAgICAgICAgICAgICAgcmVhc29uID0gImV4aGF1c3RlZCI7IGJyZWFrCiAgICAgICAgICAgIGlmIGtpbmQgPT0gUkVTRVQ6CiAgICAgICAgICAgICAgICBvYnMgPSBlbnYucmVzZXQoKTsgYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShncmlkKTsgY3VyX2tleSA9IHJvb3Rfa2V5OyBncy5wbGFuID0gW10KICAgICAgICAgICAgICAgIG1tID0gTm9uZTsgbW90aW9uX2RlYWQgPSBGYWxzZTsgdHJpZWRfdGFyZ2V0cy5jbGVhcigpOyBsZXZlbF9vZl9tb2RlbCA9IHByZXZfbGV2ZWxzCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBnYSwgZGF0YSA9IHRvX2dhbWVfYWN0aW9uKGFjdGlvbikKICAgICAgICAgICAgb2JzID0gZW52LnN0ZXAoZ2EsIGRhdGE9ZGF0YSkgaWYgZGF0YSBlbHNlIGVudi5zdGVwKGdhKQogICAgICAgICAgICByZWNvcmQoYWN0aW9uLCBvYnMpCgogICAgICAgIHJldHVybiBQbGF5UmVzdWx0KGdhbWVfaWQsIHByZXZfbGV2ZWxzLCB3aW5fbGV2ZWxzLCBhY3Rpb25zLAogICAgICAgICAgICAgICAgICAgICAgICAgIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuV0lOLCBsZW4oZ3Mud20pLCByZWFzb24pCgogICAgIyAtLS0tLSBtb3Rpb24gbGVhcm5pbmcgLS0tLS0KICAgIGRlZiBfbGVhcm5fbW90aW9uKHNlbGYsIGVudiwgb2JzLCBncmlkLCBiZywgc2ltcGxlX2F2YWlsLCByZWNvcmRfZm4pIC0+IE1WLk1vdGlvbk1vZGVsIHwgTm9uZToKICAgICAgICAiIiJUcnkgZWFjaCBzaW1wbGUgYWN0aW9uIG9uY2U7IGRldGVjdCB0aGUgYXZhdGFyIChjb25zaXN0ZW50bHktdHJhbnNsYXRpbmcgY29sb3IpLiIiIgogICAgICAgIHZvdGVzOiBkaWN0W2ludCwgZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV1dID0ge30gICMgY29sb3IgLT4ge2FjdGlvbjogKGRyLGRjKX0KICAgICAgICBjdXJfZ3JpZCA9IGdyaWQKICAgICAgICBsYXN0X29icyA9IG9icwogICAgICAgIGZvciBhaWQgaW4gc2ltcGxlX2F2YWlsOgogICAgICAgICAgICBiZWZvcmUgPSBjdXJfZ3JpZAogICAgICAgICAgICBhY3Rpb24gPSAoIlMiLCBhaWQpCiAgICAgICAgICAgIGdhLCBfID0gdG9fZ2FtZV9hY3Rpb24oYWN0aW9uKQogICAgICAgICAgICBvID0gZW52LnN0ZXAoZ2EpCiAgICAgICAgICAgIHJlY29yZF9mbihhY3Rpb24sIG8pCiAgICAgICAgICAgIGxhc3Rfb2JzID0gbwogICAgICAgICAgICBhZnRlciA9IFAudG9fZ3JpZChvLmZyYW1lKQogICAgICAgICAgICByZXMgPSBNVi5pbmZlcl90cmFuc2xhdGlvbihiZWZvcmUsIGFmdGVyLCBiZykKICAgICAgICAgICAgaWYgcmVzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgY29sb3IsIGRyLCBkYyA9IHJlcwogICAgICAgICAgICAgICAgdm90ZXMuc2V0ZGVmYXVsdChjb2xvciwge30pW2FpZF0gPSAoZHIsIGRjKQogICAgICAgICAgICBjdXJfZ3JpZCA9IGFmdGVyCiAgICAgICAgICAgIGlmIG8uc3RhdGUgaW4gKEdhbWVTdGF0ZS5XSU4sIEdhbWVTdGF0ZS5HQU1FX09WRVIpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBzZWxmLl9sYXN0X29icyA9IGxhc3Rfb2JzCiAgICAgICAgaWYgbm90IHZvdGVzOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICMgYXZhdGFyID0gY29sb3IgdGhhdCBtb3ZlZCBmb3IgdGhlIG1vc3QgYWN0aW9ucwogICAgICAgIGF2YXRhcl9jb2xvciA9IG1heCh2b3Rlcywga2V5PWxhbWJkYSBjOiBsZW4odm90ZXNbY10pKQogICAgICAgIHJldHVybiBNVi5Nb3Rpb25Nb2RlbChhdmF0YXJfY29sb3I9YXZhdGFyX2NvbG9yLCBkZWx0YXM9dm90ZXNbYXZhdGFyX2NvbG9yXSkKCiAgICBkZWYgX25leHRfdGFyZ2V0KHNlbGYsIGdyaWQsIGJnLCBtbTogTVYuTW90aW9uTW9kZWwsIHRyaWVkKSAtPiB0dXBsZVtpbnQsIGludF0gfCBOb25lOgogICAgICAgICIiIlBpY2sgdGhlIG5lYXJlc3QgdW50cmllZCBub24tYXZhdGFyIG9iamVjdCBjZW50cm9pZCB0byBuYXZpZ2F0ZSB0by4iIiIKICAgICAgICBvYmpzID0gUC5jb25uZWN0ZWRfY29tcG9uZW50cyhncmlkLCBiYWNrZ3JvdW5kPWJnKQogICAgICAgIGFjID0gbW0uYXZhdGFyX2NlbnRyb2lkKGdyaWQpCiAgICAgICAgaWYgYWMgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBjYW5kcyA9IFtdCiAgICAgICAgZm9yIG8gaW4gb2JqczoKICAgICAgICAgICAgaWYgby5jb2xvciA9PSBtbS5hdmF0YXJfY29sb3I6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByLCBjID0gaW50KHJvdW5kKG8uY2VudHJvaWRbMF0pKSwgaW50KHJvdW5kKG8uY2VudHJvaWRbMV0pKQogICAgICAgICAgICBpZiAociwgYykgaW4gdHJpZWQ6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBkID0gYWJzKHIgLSBhY1swXSkgKyBhYnMoYyAtIGFjWzFdKQogICAgICAgICAgICBjYW5kcy5hcHBlbmQoKGQsIChyLCBjKSkpCiAgICAgICAgaWYgbm90IGNhbmRzOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGNhbmRzLnNvcnQoKQogICAgICAgIHJldHVybiBjYW5kc1swXVsxXQoKICAgIGRlZiBfbmF2aWdhdGUoc2VsZiwgZW52LCBtbTogTVYuTW90aW9uTW9kZWwsIHRhcmdldCwgcmVjb3JkX2ZuLCBzdGFydF9sZXZlbHMpOgogICAgICAgICIiIkdyZWVkaWx5IGRyaXZlIHRoZSBhdmF0YXIgdG93YXJkIHRhcmdldCB1c2luZyBsZWFybmVkIGRlbHRhcy4gUmV0dXJucyBsYXN0IG9icy4iIiIKICAgICAgICBvYnMgPSBzZWxmLl9sYXN0X29icwogICAgICAgIHRyLCB0YyA9IHRhcmdldAogICAgICAgIHN0ZXBzID0gMAogICAgICAgIHN0YWxlID0gMAogICAgICAgIHdoaWxlIHN0ZXBzIDwgc2VsZi5uYXZfc3RlcF9jYXA6CiAgICAgICAgICAgIGdyaWQgPSBQLnRvX2dyaWQob2JzLmZyYW1lKQogICAgICAgICAgICBhYyA9IG1tLmF2YXRhcl9jZW50cm9pZChncmlkKQogICAgICAgICAgICBpZiBhYyBpcyBOb25lOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgY3IsIGNjID0gYWMKICAgICAgICAgICAgaWYgYWJzKGNyIC0gdHIpIDwgMSBhbmQgYWJzKGNjIC0gdGMpIDwgMToKICAgICAgICAgICAgICAgIGJyZWFrICAjIGFycml2ZWQKICAgICAgICAgICAgIyBjaG9vc2UgYWN0aW9uIG1pbmltaXppbmcgcG9zdC1tb3ZlIGRpc3RhbmNlCiAgICAgICAgICAgIGJlc3RfYSwgYmVzdF9kID0gTm9uZSwgTm9uZQogICAgICAgICAgICBjdXJfZCA9IGFicyhjciAtIHRyKSArIGFicyhjYyAtIHRjKQogICAgICAgICAgICBmb3IgYWlkLCAoZHIsIGRjKSBpbiBtbS5kZWx0YXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIG5kID0gYWJzKGNyICsgZHIgLSB0cikgKyBhYnMoY2MgKyBkYyAtIHRjKQogICAgICAgICAgICAgICAgaWYgYmVzdF9kIGlzIE5vbmUgb3IgbmQgPCBiZXN0X2Q6CiAgICAgICAgICAgICAgICAgICAgYmVzdF9kLCBiZXN0X2EgPSBuZCwgYWlkCiAgICAgICAgICAgIGlmIGJlc3RfYSBpcyBOb25lIG9yIGJlc3RfZCA+PSBjdXJfZDoKICAgICAgICAgICAgICAgIGJyZWFrICAjIG5vIGltcHJvdmluZyBtb3ZlIChncmVlZHkgc3R1Y2spCiAgICAgICAgICAgIGFjdGlvbiA9ICgiUyIsIGJlc3RfYSkKICAgICAgICAgICAgZ2EsIF8gPSB0b19nYW1lX2FjdGlvbihhY3Rpb24pCiAgICAgICAgICAgIGJlZm9yZV9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICAgICAgbyA9IGVudi5zdGVwKGdhKQogICAgICAgICAgICByZWNvcmRfZm4oYWN0aW9uLCBvKQogICAgICAgICAgICBvYnMgPSBvCiAgICAgICAgICAgIHNlbGYuX2xhc3Rfb2JzID0gbwogICAgICAgICAgICBzdGVwcyArPSAxCiAgICAgICAgICAgIGlmIG8uc3RhdGUgaW4gKEdhbWVTdGF0ZS5XSU4sIEdhbWVTdGF0ZS5HQU1FX09WRVIpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgaW50KG8ubGV2ZWxzX2NvbXBsZXRlZCBvciAwKSA+IGJlZm9yZV9sZXZlbHM6CiAgICAgICAgICAgICAgICBicmVhayAgIyByZXdhcmQhCiAgICAgICAgICAgICMgZGV0ZWN0IGJsb2NrZWQgKGF2YXRhciBkaWRuJ3QgbW92ZSkgLT4gc3RvcCB0byBhdm9pZCBzcGluCiAgICAgICAgICAgIG5nID0gUC50b19ncmlkKG8uZnJhbWUpCiAgICAgICAgICAgIG5hYyA9IG1tLmF2YXRhcl9jZW50cm9pZChuZykKICAgICAgICAgICAgaWYgbmFjIGlzIG5vdCBOb25lIGFuZCBhYnMobmFjWzBdIC0gY3IpIDwgMC41IGFuZCBhYnMobmFjWzFdIC0gY2MpIDwgMC41OgogICAgICAgICAgICAgICAgc3RhbGUgKz0gMQogICAgICAgICAgICAgICAgaWYgc3RhbGUgPj0gMjoKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc3RhbGUgPSAwCiAgICAgICAgcmV0dXJuIG9icwo=', 'policy.py': '...', 'spatial.py': 'IiIiU3BhdGlhbCBzY2VuZSBtb2RlbDogbGVhcm4gYW4gb2NjdXBhbmN5IG1hcCBvZiB0aGUgYXZhdGFyJ3Mgd29ybGQgYW5kIEEqLXBhdGhmaW5kLgoKUGFydCBvZiB0aGUgc3RydWN0dXJlZCB3b3JsZC1tb2RlbCByZWJ1aWxkLiBUaGUgYWdlbnQncyBhdmF0YXIgbW92ZXMgb24gYSBsYXR0aWNlIChlYWNoCnNpbXBsZSBhY3Rpb24gc2hpZnRzIGl0cyBjZW50cm9pZCBieSBhIHJvdWdobHktY29uc3RhbnQgZGVsdGEpLiBCeSByZWNvcmRpbmcgd2hpY2ggbGF0dGljZQpwb3NpdGlvbnMgdGhlIGF2YXRhciBzdWNjZXNzZnVsbHkgZW50ZXJlZCAoZnJlZSkgdmVyc3VzIHRyaWVkLWFuZC13YXMtYmxvY2tlZCAod2FsbCksIHdlCmJ1aWxkIGFuIG9jY3VwYW5jeSBtYXAgYW5kIHBsYW4gb3B0aW1hbCBwYXRocyB0byBhbnkgdGFyZ2V0IHdpdGggQSog4oCUIGluc3RlYWQgb2YgZ3JlZWR5Cm5hdmlnYXRpb24gdGhhdCBzdGFsbHMgYXQgdGhlIGZpcnN0IG9ic3RhY2xlLiBVbmtub3duIGNlbGxzIGFyZSB0cmVhdGVkIGFzIGZyZWUKKG9wdGltaXN0aWMpLCBzbyB0aGUgcGxhbm5lciByb3V0ZXMgYXJvdW5kICprbm93biogd2FsbHMgYW5kIHByb2JlcyB0aGUgdW5rbm93bi4KCkFzc3VtZXMgYXhpcy1hbGlnbmVkIG1vdmVtZW50ICh0aGUgZG9taW5hbnQgQVJDLUFHSS0zIGNvbnRyb2wgc2NoZW1lKS4gSWYgZGVsdGFzIGFyZW4ndApheGlzLWFsaWduZWQvY29uc2lzdGVudCwgdGhlIGNhbGxlciBzaG91bGQgbm90IHVzZSB0aGlzIGFuZCBmYWxsIGJhY2sgdG8gZ3JhcGggZXhwbG9yYXRpb24uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhlYXBxCmZyb20gbWF0aCBpbXBvcnQgZ2NkCgoKY2xhc3MgT2NjdXBhbmN5TWFwOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRlbHRhczogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV0pOgogICAgICAgICMga2VlcCBvbmx5IG5vbnplcm8sIGF4aXMtYWxpZ25lZCBkZWx0YXMgKG9uZSBheGlzIHplcm8pCiAgICAgICAgc2VsZi5kZWx0YXMgPSB7CiAgICAgICAgICAgIGE6IChkciwgZGMpIGZvciBhLCAoZHIsIGRjKSBpbiBkZWx0YXMuaXRlbXMoKQogICAgICAgICAgICBpZiAoZHIsIGRjKSAhPSAoMCwgMCkgYW5kIChkciA9PSAwIG9yIGRjID09IDApCiAgICAgICAgfQogICAgICAgIG1hZ3MgPSBbYWJzKGRyKSBvciBhYnMoZGMpIGZvciBkciwgZGMgaW4gc2VsZi5kZWx0YXMudmFsdWVzKCldCiAgICAgICAgc2VsZi5zdGVwID0gX2djZF9saXN0KG1hZ3MpIGlmIG1hZ3MgZWxzZSAxCiAgICAgICAgc2VsZi5vcmlnaW46IHR1cGxlW2Zsb2F0LCBmbG9hdF0gfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuZnJlZTogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgICAgIHNlbGYuYmxvY2tlZDogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHVzYWJsZShzZWxmKSAtPiBib29sOgogICAgICAgICMgbmVlZCBheGlzLWFsaWduZWQgbW92ZXMgY292ZXJpbmcgYm90aCBheGVzIHRvIHBhdGhmaW5kIGluIDJECiAgICAgICAgaGF2ZXNfcm93ID0gYW55KGRyICE9IDAgZm9yIGRyLCBkYyBpbiBzZWxmLmRlbHRhcy52YWx1ZXMoKSkKICAgICAgICBoYXZlc19jb2wgPSBhbnkoZGMgIT0gMCBmb3IgZHIsIGRjIGluIHNlbGYuZGVsdGFzLnZhbHVlcygpKQogICAgICAgIHJldHVybiBzZWxmLnN0ZXAgPiAwIGFuZCBoYXZlc19yb3cgYW5kIGhhdmVzX2NvbAoKICAgIGRlZiBxdWFudGl6ZShzZWxmLCBjZW50cm9pZDogdHVwbGVbZmxvYXQsIGZsb2F0XSkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIGlmIHNlbGYub3JpZ2luIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYub3JpZ2luID0gY2VudHJvaWQKICAgICAgICByMCwgYzAgPSBzZWxmLm9yaWdpbgogICAgICAgIHJldHVybiAocm91bmQoKGNlbnRyb2lkWzBdIC0gcjApIC8gc2VsZi5zdGVwKSwgcm91bmQoKGNlbnRyb2lkWzFdIC0gYzApIC8gc2VsZi5zdGVwKSkKCiAgICBkZWYgX3N0ZXBfdW5pdHMoc2VsZiwgZHI6IGludCwgZGM6IGludCkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHJldHVybiAoaW50KHJvdW5kKGRyIC8gc2VsZi5zdGVwKSksIGludChyb3VuZChkYyAvIHNlbGYuc3RlcCkpKQoKICAgIGRlZiBvYnNlcnZlX21vdmUoc2VsZiwgYmVmb3JlOiB0dXBsZVtmbG9hdCwgZmxvYXRdLCBhY3Rpb25faWQ6IGludCwKICAgICAgICAgICAgICAgICAgICAgYWZ0ZXI6IHR1cGxlW2Zsb2F0LCBmbG9hdF0pIC0+IE5vbmU6CiAgICAgICAgIiIiUmVjb3JkIHRoZSBvdXRjb21lIG9mIGEgc2ltcGxlIG1vdmUgZm9yIG9jY3VwYW5jeSBsZWFybmluZy4iIiIKICAgICAgICBpZiBhY3Rpb25faWQgbm90IGluIHNlbGYuZGVsdGFzOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBxYiA9IHNlbGYucXVhbnRpemUoYmVmb3JlKQogICAgICAgIHFhID0gc2VsZi5xdWFudGl6ZShhZnRlcikKICAgICAgICBzZWxmLmZyZWUuYWRkKHFiKQogICAgICAgIGRyLCBkYyA9IHNlbGYuZGVsdGFzW2FjdGlvbl9pZF0KICAgICAgICB1ciwgdWMgPSBzZWxmLl9zdGVwX3VuaXRzKGRyLCBkYykKICAgICAgICB0YXJnZXQgPSAocWJbMF0gKyB1ciwgcWJbMV0gKyB1YykKICAgICAgICBpZiBxYSA9PSBxYjoKICAgICAgICAgICAgIyBhdmF0YXIgZGlkbid0IG1vdmUgLT4gdGhlIHRhcmdldCBjZWxsIGlzIGJsb2NrZWQgKHdhbGwvYm91bmRhcnkpCiAgICAgICAgICAgIHNlbGYuYmxvY2tlZC5hZGQodGFyZ2V0KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuZnJlZS5hZGQocWEpCgogICAgZGVmIGFzdGFyKHNlbGYsIHN0YXJ0OiB0dXBsZVtmbG9hdCwgZmxvYXRdLCBnb2FsOiB0dXBsZVtmbG9hdCwgZmxvYXRdKSAtPiBsaXN0W2ludF0gfCBOb25lOgogICAgICAgICIiIlJldHVybiBhIGxpc3Qgb2YgYWN0aW9uX2lkcyBtb3ZpbmcgdGhlIGF2YXRhciBmcm9tIHN0YXJ0IHRvIGdvYWwsIG9yIE5vbmUuCgogICAgICAgIFBsYW5zIG92ZXIgdGhlIGxhdHRpY2U6IGtub3duLWJsb2NrZWQgY2VsbHMgYXJlIHdhbGxzOyB1bmtub3duIGNlbGxzIGFyZSBhc3N1bWVkCiAgICAgICAgZnJlZSAob3B0aW1pc3RpYykuIEdvYWwgaXMgbWF0Y2hlZCBhdCBsYXR0aWNlIHJlc29sdXRpb24uCiAgICAgICAgIiIiCiAgICAgICAgaWYgbm90IHNlbGYudXNhYmxlOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHFzID0gc2VsZi5xdWFudGl6ZShzdGFydCkKICAgICAgICBxZyA9IHNlbGYucXVhbnRpemUoZ29hbCkKICAgICAgICBpZiBxcyA9PSBxZzoKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgbW92ZXMgPSBbKGEsIHNlbGYuX3N0ZXBfdW5pdHMoZHIsIGRjKSkgZm9yIGEsIChkciwgZGMpIGluIHNlbGYuZGVsdGFzLml0ZW1zKCldCgogICAgICAgIGRlZiBoKHApOgogICAgICAgICAgICByZXR1cm4gYWJzKHBbMF0gLSBxZ1swXSkgKyBhYnMocFsxXSAtIHFnWzFdKQoKICAgICAgICBvcGVuaCA9IFsoaChxcyksIDAsIHFzLCBbXSldCiAgICAgICAgc2VlbiA9IHtxczogMH0KICAgICAgICBib3VuZCA9IDQgKiAoYWJzKHFzWzBdIC0gcWdbMF0pICsgYWJzKHFzWzFdIC0gcWdbMV0pICsgNCkgICMgYXZvaWQgcnVuYXdheSBpbiBvcGVuIHNwYWNlCiAgICAgICAgd2hpbGUgb3Blbmg6CiAgICAgICAgICAgIGYsIGcsIHBvcywgcGF0aCA9IGhlYXBxLmhlYXBwb3Aob3BlbmgpCiAgICAgICAgICAgIGlmIHBvcyA9PSBxZzoKICAgICAgICAgICAgICAgIHJldHVybiBwYXRoCiAgICAgICAgICAgIGlmIGcgPiBib3VuZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhLCAodXIsIHVjKSBpbiBtb3ZlczoKICAgICAgICAgICAgICAgIG5wb3MgPSAocG9zWzBdICsgdXIsIHBvc1sxXSArIHVjKQogICAgICAgICAgICAgICAgaWYgbnBvcyBpbiBzZWxmLmJsb2NrZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG5nID0gZyArIDEKICAgICAgICAgICAgICAgIGlmIG5wb3MgaW4gc2VlbiBhbmQgc2VlbltucG9zXSA8PSBuZzoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2VlbltucG9zXSA9IG5nCiAgICAgICAgICAgICAgICBoZWFwcS5oZWFwcHVzaChvcGVuaCwgKG5nICsgaChucG9zKSwgbmcsIG5wb3MsIHBhdGggKyBbYV0pKQogICAgICAgIHJldHVybiBOb25lCgoKZGVmIF9nY2RfbGlzdCh4czogbGlzdFtpbnRdKSAtPiBpbnQ6CiAgICBnID0gMAogICAgZm9yIHggaW4geHM6CiAgICAgICAgZyA9IGdjZChnLCBpbnQoeCkpCiAgICByZXR1cm4gZyBvciAxCg==', 'salience_explorer.py': 'IiIiU2FsaWVuY2VFeHBsb3JlciDigJQgb3VyIG93biByZWltcGxlbWVudGF0aW9uIG9mIHRoZSBwdWJsaXNoZWQgaGllcmFyY2hpY2FsIHNhbGllbmNlLXRpZXJlZApncmFwaC1leHBsb3JhdGlvbiBhbGdvcml0aG0gKGFyWGl2IDI1MTIuMjQxNTYgImp1c3QtZXhwbG9yZSIsIEFsZ29yaXRobSAxKS4KClRoaXMgaXMgYSBjbGVhbi1yb29tIHJlaW1wbGVtZW50YXRpb24gb2YgdGhlICphbGdvcml0aG0gYXMgZGVzY3JpYmVkIGluIHRoZSBwYXBlciogKE5PVCBhCmNvcHkgb2YgdGhlaXIgY29kZSk6IGEgcHVyZSBvYmplY3QtZ3JhcGggZXhwbG9yZXIgd2l0aCBubyBtb3Rpb24gbW9kZWwsIHdoaWNoIGlzIHdoeSBpdAphdm9pZHMgb3VyIGh5YnJpZCdzIHNva29iYW4tZnJhZ2lsaXR5LiBSZWFjdGl2ZSBvbmUtYWN0aW9uLXBlci1jYWxsIGludGVyZmFjZSAoc2FtZSBhcwpIeWJyaWRQb2xpY3kpIHNvIGl0IGRyb3BzIGludG8gcnVuX3JlYWN0aXZlIC8gdGhlIHN1Ym1pc3Npb24gYWRhcHRlci4gVGhlIGJhbmtlZCByZWFjdGl2ZQphZ2VudCBpcyB1bnRvdWNoZWQ7IHRoaXMgaXMgYSBzZXBhcmF0ZSBwb2xpY3kgbWVhc3VyZWQgaGVhZC10by1oZWFkLgoKQWxnb3JpdGhtIDEgKGhpZXJhcmNoaWNhbCBhY3Rpb24gc2VsZWN0aW9uKSwgcGVyIHN0YXRlIG5vZGUsIGF0IHNhbGllbmNlIHRocmVzaG9sZCBwOgogIDEpIGlmIGEga25vd24gYWN0aW9uIGhlcmUgcHJvZHVjZWQgcmV3YXJkLCB0YWtlIGl0IChleHBsb2l0KTsKICAyKSBlbHNlIGlmIHRoaXMgbm9kZSBoYXMgYW4gdW50ZXN0ZWQgYWN0aW9uIHdpdGggdGllciA8PSBwLCB0YWtlIG9uZSBVTklGT1JNTFkgQVQgUkFORE9NCiAgICAgYW1vbmcgdGhlIGxvd2VzdCBzdWNoIHRpZXI7CiAgMykgZWxzZSBtb3ZlIGFsb25nIHRoZSBzaG9ydGVzdCBrbm93biBwYXRoIHRvIHRoZSBuZWFyZXN0IHJlYWNoYWJsZSBub2RlIHRoYXQgc3RpbGwgaGFzCiAgICAgYW4gdW50ZXN0ZWQgYWN0aW9uIHdpdGggdGllciA8PSBwOwogIDQpIGVsc2UgcmFpc2UgcCBhbmQgcmVjdXJzZTsgaWYgcCBleGhhdXN0ZWQsIFJFU0VUIHRvIHJvb3QgKGJvdW5kZWQpLCB0aGVuIHN0b3AuClN0YXRlIGlkID0gb2JqZWN0LXN0cnVjdHVyZSBoYXNoIHdpdGggZnJlcXVlbnRseS1jaGFuZ2luZyAoc3RhdHVzLWJhci9jb3VudGVyKSBjZWxscyBtYXNrZWQuCkFjdGlvbnMgYXJlIHNhbGllbmNlLXRpZXJlZDogc2ltcGxlIGFjdGlvbnMgdGllciAwOyBjbGlja3MgYnkgb2JqZWN0IHNhbGllbmNlICgwLi45KS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4gaW1wb3J0IHBlcmNlcHRpb24gYXMgUAoKU0lNUExFX0lEUyA9IFsxLCAyLCAzLCA0LCA1XQpNQVhfVElFUiA9IDkKCgpjbGFzcyBfTm9kZToKICAgIF9fc2xvdHNfXyA9ICgia2V5IiwgImNhbmRzIiwgInRpZXIiLCAiZWRnZXMiLCAidGVybWluYWwiLCAidmlzaXRzIikKCiAgICBkZWYgX19pbml0X18oc2VsZiwga2V5LCBjYW5kc193aXRoX3RpZXJzLCB0ZXJtaW5hbD1GYWxzZSk6CiAgICAgICAgc2VsZi5rZXkgPSBrZXkKICAgICAgICBzZWxmLmNhbmRzID0gdHVwbGUoYSBmb3IgYSwgX3QgaW4gY2FuZHNfd2l0aF90aWVycykKICAgICAgICBzZWxmLnRpZXIgPSB7YTogdCBmb3IgYSwgdCBpbiBjYW5kc193aXRoX3RpZXJzfQogICAgICAgIHNlbGYuZWRnZXM6IGRpY3QgPSB7fSAgIyBhY3Rpb24gLT4gKG5leHRfa2V5LCByZXdhcmQpCiAgICAgICAgc2VsZi50ZXJtaW5hbCA9IHRlcm1pbmFsCiAgICAgICAgc2VsZi52aXNpdHMgPSAwCgogICAgZGVmIHVudHJpZWRfbGUoc2VsZiwgcCk6CiAgICAgICAgcmV0dXJuIFthIGZvciBhIGluIHNlbGYuY2FuZHMgaWYgYSBub3QgaW4gc2VsZi5lZGdlcyBhbmQgc2VsZi50aWVyLmdldChhLCAwKSA8PSBwXQoKICAgIGRlZiBoYXNfdW50cmllZF9sZShzZWxmLCBwKToKICAgICAgICByZXR1cm4gbm90IHNlbGYudGVybWluYWwgYW5kIGFueSgKICAgICAgICAgICAgYSBub3QgaW4gc2VsZi5lZGdlcyBhbmQgc2VsZi50aWVyLmdldChhLCAwKSA8PSBwIGZvciBhIGluIHNlbGYuY2FuZHMpCgogICAgZGVmIHJld2FyZF9hY3Rpb24oc2VsZik6CiAgICAgICAgYmVzdCwgYnIgPSBOb25lLCAwLjAKICAgICAgICBmb3IgYSwgKF9uaywgcikgaW4gc2VsZi5lZGdlcy5pdGVtcygpOgogICAgICAgICAgICBpZiByID4gYnI6CiAgICAgICAgICAgICAgICBiZXN0LCBiciA9IGEsIHIKICAgICAgICByZXR1cm4gYmVzdAoKCmNsYXNzIFNhbGllbmNlRXhwbG9yZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbWF4X2NsaWNrX3RhcmdldHM6IGludCA9IDk2LCBzZWVkOiBpbnQgPSAwLAogICAgICAgICAgICAgICAgIG1heF9zdHVja19yZXNldHM6IGludCA9IDIwMCwgdHJ1c3RfdGhyZXNob2xkOiBpbnQgPSAzLAogICAgICAgICAgICAgICAgIGJvcmRlcl9tYXNrOiBpbnQgPSAyKSAtPiBOb25lOgogICAgICAgIHNlbGYubWF4X2NsaWNrX3RhcmdldHMgPSBtYXhfY2xpY2tfdGFyZ2V0cwogICAgICAgIHNlbGYucm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICAgICAgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzID0gbWF4X3N0dWNrX3Jlc2V0cwogICAgICAgICMgYm9yZGVyX21hc2sgPiAwIGVuYWJsZXMgdGhlIGR5bmFtaWMtYm9yZGVyIChIVUQvcHJvZ3Jlc3MtYmFyKSBtYXNrOiBjZWxscyB3aXRoaW4KICAgICAgICAjIHRoaXMgbWFueSByb3dzL2NvbHMgb2YgdGhlIGdyaWQgZWRnZSB0aGF0IGhhdmUgRVZFUiBjaGFuZ2VkIGFyZSBkcm9wcGVkIGZyb20gdGhlCiAgICAgICAgIyBzdGF0ZSBrZXkuIE1vbm90b25pYyBib3R0b20tZWRnZSBwcm9ncmVzcyBiYXJzIChyZTg2L3dhMzApIGNoYW5nZSBlYWNoIGNlbGwgb25seQogICAgICAgICMgb25jZSwgc28gdGhlIGNlbGwtZnJlcXVlbmN5IFZvbGF0aWxpdHlUcmFja2VyIG5ldmVyIGNhdGNoZXMgdGhlbSAtPiBldmVyeSBzdGF0ZSBpcwogICAgICAgICMgZm9yZXZlci11bmlxdWUgLT4gZ3JhcGggZXhwbG9kZXMgKHJlODYgMS4xIGFjdC9zdGF0ZSkuIE1hc2tpbmcgdGhlIGR5bmFtaWMgZWRnZSBiYW5kCiAgICAgICAgIyByZXN0b3JlcyBzdGF0ZSByZXZpc2l0cyB3aXRob3V0IHRvdWNoaW5nIHRoZSBpbnRlcmlvciBwbGF5IGFyZWEuIDAgPT0gb2ZmLiBEZWZhdWx0IDIKICAgICAgICAjIGNhdGNoZXMgMi13aWRlIGVkZ2UgYmFycyAoc2MyNSByaWdodC1lZGdlIGNvbHMgNjItNjMsIHNrNDgpIHRoYXQgYmFuZD0xIGhhbGYtbWFza3M7CiAgICAgICAgIyBiYW5kPTIgbWVhc3VyZWQgMzMgbGV2ZWxzIEAzMGsgKHN0cmljdCBzdXBlcnNldCBvZiBiYW5kPTEncyAzMiwgK3NrNDgsIG5vIHJlZ3Jlc3Npb25zKS4KICAgICAgICBzZWxmLmJvcmRlcl9tYXNrID0gbWF4KDAsIGludChib3JkZXJfbWFzaykpCiAgICAgICAgIyB0cnVzdF90aHJlc2hvbGQgPiAxIGVuYWJsZXMgc3VzcGljaW91cy10cmFuc2l0aW9uIGZpbHRlcmluZzogYSBORVcgdHJhbnNpdGlvbiB0aGF0CiAgICAgICAgIyBjb25mbGljdHMgd2l0aCBhbiBhbHJlYWR5LXJlY29yZGVkIGVkZ2UgKHRoZSBzaWduYXR1cmUgb2YgYW5pbWF0aW9uL2ZyYW1lIG5vaXNlIG9uCiAgICAgICAgIyByZWFsIGdhbWVzKSBtdXN0IHJlcGVhdCB0aGlzIG1hbnkgdGltZXMgYmVmb3JlIGl0IG92ZXJ3cml0ZXMgdGhlIHRydXN0ZWQgZWRnZS4gVGhlCiAgICAgICAgIyBmaXJzdCBvYnNlcnZhdGlvbiBvZiBhbnkgZWRnZSwgYW5kIGFueSByZXdhcmQtYmVhcmluZyB0cmFuc2l0aW9uLCBpcyB0cnVzdGVkIGF0IG9uY2UKICAgICAgICAjIChzbyBkZXRlcm1pbmlzdGljIGdhbWVzIGFyZSBub3Qgc2xvd2VkKS4gdHJ1c3RfdGhyZXNob2xkID09IDEgPT0gb3JpZ2luYWwgYmVoYXZpb3VyLgogICAgICAgIHNlbGYudHJ1c3RfdGhyZXNob2xkID0gbWF4KDEsIGludCh0cnVzdF90aHJlc2hvbGQpKQogICAgICAgIHNlbGYucmVzZXRfYWxsKCkKCiAgICAjIGV4cG9zZSAuZ3Mud20tbGlrZSBsZW5ndGggZm9yIHRoZSBydW5uZXIncyBzdGF0ZXNfc2VlbiAoZHVjay10eXBpbmcpCiAgICBAcHJvcGVydHkKICAgIGRlZiBncyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZgoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHdtKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLm5vZGVzCgogICAgZGVmIHJlc2V0X2FsbChzZWxmKToKICAgICAgICBzZWxmLnZ0ID0gUC5Wb2xhdGlsaXR5VHJhY2tlcigpCiAgICAgICAgc2VsZi5ub2RlczogZGljdFtieXRlcywgX05vZGVdID0ge30KICAgICAgICBzZWxmLmJnOiBpbnQgfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYucm9vdF9rZXk6IGJ5dGVzIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLmFjdGl2ZV9ncm91cCA9IDAKICAgICAgICBzZWxmLnBsYW46IGxpc3QgPSBbXQogICAgICAgIHNlbGYuX2V4cGVjdDogYnl0ZXMgfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYucHJldl9rZXk6IGJ5dGVzIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLnByZXZfYWN0aW9uID0gTm9uZQogICAgICAgIHNlbGYucHJldl9sZXZlbHMgPSAwCiAgICAgICAgc2VsZi5leHBlY3RfcmVzZXQgPSBGYWxzZQogICAgICAgIHNlbGYuc3R1Y2tfcmVzZXRzID0gMAogICAgICAgIHNlbGYucGVuZGluZzogZGljdCA9IHt9ICAjIChrZXksIGFjdGlvbikgLT4gKGNhbmRpZGF0ZV9uZXh0X2tleSwgY291bnQpIGZvciBzdXNwaWNpb24gZmlsdGVyCgogICAgZGVmIF9jYW5kaWRhdGVzKHNlbGYsIGdyaWQsIGF2YWlsYWJsZSk6CiAgICAgICAgY2FuZHMgPSBbXQogICAgICAgIGZvciBhaWQgaW4gU0lNUExFX0lEUzoKICAgICAgICAgICAgaWYgYWlkIGluIGF2YWlsYWJsZToKICAgICAgICAgICAgICAgIGNhbmRzLmFwcGVuZCgoKCJTIiwgYWlkKSwgMCkpCiAgICAgICAgaWYgNiBpbiBhdmFpbGFibGU6CiAgICAgICAgICAgIGZvciB4LCB5LCBwcmlvIGluIFAuc2FsaWVudF9jbGlja190YXJnZXRzKGdyaWQsIG1heF90YXJnZXRzPXNlbGYubWF4X2NsaWNrX3RhcmdldHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvYXJzZV9ncmlkX3N0ZXA9OCk6CiAgICAgICAgICAgICAgICBjYW5kcy5hcHBlbmQoKCgiQyIsIGludCh4KSwgaW50KHkpKSwgaW50KHByaW8pKSkKICAgICAgICByZXR1cm4gY2FuZHMKCiAgICBkZWYgX2tleShzZWxmLCBncmlkKToKICAgICAgICBtID0gc2VsZi52dC5tYXNrKCkKICAgICAgICBibSA9IHNlbGYuX2JvcmRlcl9tYXNrKCkKICAgICAgICBpZiBibSBpcyBub3QgTm9uZToKICAgICAgICAgICAgbSA9IG0gfCBibQogICAgICAgIGlmIG0uYW55KCk6CiAgICAgICAgICAgIGdyaWQgPSBncmlkLmNvcHkoKQogICAgICAgICAgICBncmlkW21dID0gc2VsZi5iZyBpZiBzZWxmLmJnIGlzIG5vdCBOb25lIGVsc2UgMAogICAgICAgIHJldHVybiBQLm9iamVjdF9zdGF0ZV9rZXkoZ3JpZCwgYmFja2dyb3VuZD1zZWxmLmJnKQoKICAgIGRlZiBfYm9yZGVyX21hc2soc2VsZik6CiAgICAgICAgIiIiRWRnZSBjZWxscyAod2l0aGluIGJvcmRlcl9tYXNrIG9mIHRoZSBncmlkIGVkZ2UpIHRoYXQgaGF2ZSBldmVyIGNoYW5nZWQgLT4gSFVELiIiIgogICAgICAgIGIgPSBzZWxmLmJvcmRlcl9tYXNrCiAgICAgICAgaWYgYiA8PSAwIG9yIHNlbGYudnQuY2hhbmdlcyBpcyBOb25lIG9yIHNlbGYudnQuc3RlcHMgPCBzZWxmLnZ0Lm1pbl9zdGVwczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBlZGdlID0gbnAuemVyb3Moc2VsZi52dC5zaGFwZSwgZHR5cGU9Ym9vbCkKICAgICAgICBlZGdlWzpiXSA9IGVkZ2VbLWI6XSA9IFRydWUKICAgICAgICBlZGdlWzosIDpiXSA9IGVkZ2VbOiwgLWI6XSA9IFRydWUKICAgICAgICByZXR1cm4gZWRnZSAmIChzZWxmLnZ0LmNoYW5nZXMgPiAwKQoKICAgIGRlZiBfb2JzZXJ2ZShzZWxmLCBrZXksIGNhbmRzLCB0ZXJtaW5hbD1GYWxzZSk6CiAgICAgICAgbiA9IHNlbGYubm9kZXMuZ2V0KGtleSkKICAgICAgICBpZiBuIGlzIE5vbmU6CiAgICAgICAgICAgIG4gPSBfTm9kZShrZXksIGNhbmRzLCB0ZXJtaW5hbCkKICAgICAgICAgICAgc2VsZi5ub2Rlc1trZXldID0gbgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG4udGVybWluYWwgPSBuLnRlcm1pbmFsIG9yIHRlcm1pbmFsCiAgICAgICAgbi52aXNpdHMgKz0gMQogICAgICAgIHJldHVybiBuCgogICAgZGVmIF9wYXRoX3RvX2Zyb250aWVyKHNlbGYsIHN0YXJ0LCBwKToKICAgICAgICBpZiBzdGFydCBub3QgaW4gc2VsZi5ub2RlczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiBzZWxmLm5vZGVzW3N0YXJ0XS5oYXNfdW50cmllZF9sZShwKToKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgc2VlbiA9IHtzdGFydH0KICAgICAgICBxID0gZGVxdWUoWyhzdGFydCwgW10pXSkKICAgICAgICB3aGlsZSBxOgogICAgICAgICAgICBrLCBwYXRoID0gcS5wb3BsZWZ0KCkKICAgICAgICAgICAgbm9kZSA9IHNlbGYubm9kZXMuZ2V0KGspCiAgICAgICAgICAgIGlmIG5vdCBub2RlOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZm9yIGEsIChuaywgX3IpIGluIG5vZGUuZWRnZXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIG5rIGluIHNlZW46CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHNlZW4uYWRkKG5rKQogICAgICAgICAgICAgICAgbnBfID0gcGF0aCArIFthXQogICAgICAgICAgICAgICAgbm4gPSBzZWxmLm5vZGVzLmdldChuaykKICAgICAgICAgICAgICAgIGlmIG5uIGlzIG5vdCBOb25lIGFuZCBubi5oYXNfdW50cmllZF9sZShwKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gbnBfCiAgICAgICAgICAgICAgICBxLmFwcGVuZCgobmssIG5wXykpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgZGVjaWRlKHNlbGYsIGdyaWQsIGdzdGF0ZV90ZXJtaW5hbCwgZ3N0YXRlX25vdHBsYXllZCwgbGV2ZWxzLCBhdmFpbGFibGUpOgogICAgICAgIHNlbGYudnQudXBkYXRlKGdyaWQpCiAgICAgICAgaWYgc2VsZi5iZyBpcyBOb25lOgogICAgICAgICAgICBzZWxmLmJnID0gUC5kZXRlY3RfYmFja2dyb3VuZChncmlkKQogICAgICAgIGN1ciA9IHNlbGYuX2tleShncmlkKQoKICAgICAgICBpZiBnc3RhdGVfdGVybWluYWwgb3IgZ3N0YXRlX25vdHBsYXllZDoKICAgICAgICAgICAgaWYgZ3N0YXRlX3Rlcm1pbmFsIGFuZCBzZWxmLnByZXZfYWN0aW9uIGlzIG5vdCBOb25lIGFuZCBzZWxmLnByZXZfa2V5IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgc2VsZi5fcmVjb3JkKHNlbGYucHJldl9rZXksIHNlbGYucHJldl9hY3Rpb24sIGN1ciwgMC4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuX2NhbmRpZGF0ZXMoZ3JpZCwgYXZhaWxhYmxlKSwgdGVybWluYWw9VHJ1ZSkKICAgICAgICAgICAgc2VsZi5wcmV2X2FjdGlvbiA9IE5vbmUKICAgICAgICAgICAgc2VsZi5leHBlY3RfcmVzZXQgPSBUcnVlCiAgICAgICAgICAgIHNlbGYucGxhbiA9IFtdCiAgICAgICAgICAgIHJldHVybiAoInJlc2V0IiwpCgogICAgICAgIGlmIHNlbGYucm9vdF9rZXkgaXMgTm9uZToKICAgICAgICAgICAgc2VsZi5yb290X2tleSA9IGN1cgogICAgICAgICAgICBzZWxmLl9vYnNlcnZlKGN1ciwgc2VsZi5fY2FuZGlkYXRlcyhncmlkLCBhdmFpbGFibGUpKQoKICAgICAgICBpZiBzZWxmLmV4cGVjdF9yZXNldDoKICAgICAgICAgICAgc2VsZi5leHBlY3RfcmVzZXQgPSBGYWxzZQogICAgICAgICAgICBzZWxmLnByZXZfYWN0aW9uID0gTm9uZQoKICAgICAgICBpZiBzZWxmLnByZXZfYWN0aW9uIGlzIG5vdCBOb25lIGFuZCBzZWxmLnByZXZfa2V5IGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXdhcmQgPSBmbG9hdChsZXZlbHMgLSBzZWxmLnByZXZfbGV2ZWxzKQogICAgICAgICAgICBzZWxmLl9yZWNvcmQoc2VsZi5wcmV2X2tleSwgc2VsZi5wcmV2X2FjdGlvbiwgY3VyLCByZXdhcmQsCiAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kaWRhdGVzKGdyaWQsIGF2YWlsYWJsZSksIHRlcm1pbmFsPUZhbHNlKQogICAgICAgICAgICBpZiByZXdhcmQgPiAwOgogICAgICAgICAgICAgICAgc2VsZi5hY3RpdmVfZ3JvdXAgPSAwICAjIHJlLXByaW9yaXRpc2UgaGlnaCBzYWxpZW5jZSBhZnRlciBhIGxldmVsLXVwCgogICAgICAgIHNlbGYucHJldl9sZXZlbHMgPSBsZXZlbHMKICAgICAgICBhY3Rpb24gPSBzZWxmLl9jaG9vc2UoY3VyKQogICAgICAgIHNlbGYucHJldl9rZXkgPSBjdXIKICAgICAgICBzZWxmLnByZXZfYWN0aW9uID0gTm9uZSBpZiBhY3Rpb25bMF0gPT0gInJlc2V0IiBlbHNlIGFjdGlvbgogICAgICAgIHJldHVybiBhY3Rpb24KCiAgICBkZWYgX3JlY29yZChzZWxmLCBrZXksIGFjdGlvbiwgbmV4dF9rZXksIHJld2FyZCwgY2FuZHMsIHRlcm1pbmFsKToKICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KSBvciBzZWxmLl9vYnNlcnZlKGtleSwgY2FuZHMpCiAgICAgICAgZXhpc3RpbmcgPSBub2RlLmVkZ2VzLmdldChhY3Rpb24pCiAgICAgICAgaWYgKHNlbGYudHJ1c3RfdGhyZXNob2xkIDw9IDEgb3IgcmV3YXJkID4gMCBvciBleGlzdGluZyBpcyBOb25lCiAgICAgICAgICAgICAgICBvciBleGlzdGluZ1swXSA9PSBuZXh0X2tleSk6CiAgICAgICAgICAgICMgdHJ1c3QgYXQgb25jZTogZmlsdGVyaW5nIG9mZiwgcmV3YXJkLWJlYXJpbmcsIGZpcnN0IG9ic2VydmF0aW9uLCBvciBjb25zaXN0ZW50CiAgICAgICAgICAgIG5vZGUuZWRnZXNbYWN0aW9uXSA9IChuZXh0X2tleSwgcmV3YXJkKQogICAgICAgICAgICBzZWxmLnBlbmRpbmcucG9wKChrZXksIGFjdGlvbiksIE5vbmUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgIyBjb25mbGljdCB3aXRoIGEgdHJ1c3RlZCBlZGdlIC0+IHJlcXVpcmUgdGhlIG5ldyB0YXJnZXQgdG8gcmVwZWF0IGJlZm9yZSBvdmVyd3JpdGluZwogICAgICAgICAgICBwayA9IChrZXksIGFjdGlvbikKICAgICAgICAgICAgY2FuZCwgY250ID0gc2VsZi5wZW5kaW5nLmdldChwaywgKG5leHRfa2V5LCAwKSkKICAgICAgICAgICAgY2FuZCwgY250ID0gKG5leHRfa2V5LCBjbnQgKyAxKSBpZiBjYW5kID09IG5leHRfa2V5IGVsc2UgKG5leHRfa2V5LCAxKQogICAgICAgICAgICBpZiBjbnQgPj0gc2VsZi50cnVzdF90aHJlc2hvbGQ6CiAgICAgICAgICAgICAgICBub2RlLmVkZ2VzW2FjdGlvbl0gPSAobmV4dF9rZXksIHJld2FyZCkKICAgICAgICAgICAgICAgIHNlbGYucGVuZGluZy5wb3AocGssIE5vbmUpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzZWxmLnBlbmRpbmdbcGtdID0gKGNhbmQsIGNudCkKICAgICAgICBzZWxmLl9vYnNlcnZlKG5leHRfa2V5LCBjYW5kcywgdGVybWluYWw9dGVybWluYWwpCiAgICAgICAgc2VsZi5fZXhwZWN0ID0gbmV4dF9rZXkgaWYgc2VsZi5wbGFuIGVsc2UgTm9uZQoKICAgIGRlZiBfY2hvb3NlKHNlbGYsIGN1cik6CiAgICAgICAgbm9kZSA9IHNlbGYubm9kZXMuZ2V0KGN1cikKICAgICAgICBpZiBub2RlIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9yYW5kb20oY3VyKQogICAgICAgICMgMSkgZXhwbG9pdCByZXdhcmQKICAgICAgICByYSA9IG5vZGUucmV3YXJkX2FjdGlvbigpCiAgICAgICAgaWYgcmEgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiByYQogICAgICAgICMgMikgYWN0aXZlIHBsYW4gcmVwbGF5CiAgICAgICAgaWYgc2VsZi5wbGFuOgogICAgICAgICAgICBpZiBzZWxmLl9leHBlY3QgaXMgbm90IE5vbmUgYW5kIGN1ciAhPSBzZWxmLl9leHBlY3Q6CiAgICAgICAgICAgICAgICBzZWxmLnBsYW4gPSBbXQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYucGxhbi5wb3AoMCkKICAgICAgICAjIDMpIGhpZXJhcmNoaWNhbCB0aWVyIGV4cGxvcmF0aW9uCiAgICAgICAgZyA9IHNlbGYuYWN0aXZlX2dyb3VwCiAgICAgICAgd2hpbGUgZyA8PSBNQVhfVElFUjoKICAgICAgICAgICAgbG9jYWwgPSBub2RlLnVudHJpZWRfbGUoZykKICAgICAgICAgICAgaWYgbG9jYWw6CiAgICAgICAgICAgICAgICBzZWxmLmFjdGl2ZV9ncm91cCA9IGcKICAgICAgICAgICAgICAgIG1wID0gbWluKG5vZGUudGllci5nZXQoYSwgMCkgZm9yIGEgaW4gbG9jYWwpCiAgICAgICAgICAgICAgICBjaG9pY2VzID0gW2EgZm9yIGEgaW4gbG9jYWwgaWYgbm9kZS50aWVyLmdldChhLCAwKSA9PSBtcF0KICAgICAgICAgICAgICAgIHJldHVybiBjaG9pY2VzW2ludChzZWxmLnJuZy5pbnRlZ2VycygwLCBsZW4oY2hvaWNlcykpKV0KICAgICAgICAgICAgcGF0aCA9IHNlbGYuX3BhdGhfdG9fZnJvbnRpZXIoY3VyLCBnKQogICAgICAgICAgICBpZiBwYXRoOgogICAgICAgICAgICAgICAgc2VsZi5hY3RpdmVfZ3JvdXAgPSBnCiAgICAgICAgICAgICAgICBzZWxmLnBsYW4gPSBwYXRoCiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5wbGFuLnBvcCgwKQogICAgICAgICAgICBnICs9IDEKICAgICAgICAjIDQpIGV4aGF1c3RlZCBmcm9tIGhlcmUgLT4gYm91bmNlIG9mZiByb290CiAgICAgICAgaWYgc2VsZi5yb290X2tleSBpcyBub3QgTm9uZSBhbmQgY3VyICE9IHNlbGYucm9vdF9rZXkgYW5kIHNlbGYuc3R1Y2tfcmVzZXRzIDwgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzOgogICAgICAgICAgICBzZWxmLnN0dWNrX3Jlc2V0cyArPSAxCiAgICAgICAgICAgIHNlbGYuZXhwZWN0X3Jlc2V0ID0gVHJ1ZQogICAgICAgICAgICBzZWxmLnBsYW4gPSBbXQogICAgICAgICAgICByZXR1cm4gKCJyZXNldCIsKQogICAgICAgIHJldHVybiBzZWxmLl9yYW5kb20oY3VyKQoKICAgIGRlZiBfcmFuZG9tKHNlbGYsIGN1cik6CiAgICAgICAgbm9kZSA9IHNlbGYubm9kZXMuZ2V0KGN1cikKICAgICAgICBjYW5kcyA9IG5vZGUuY2FuZHMgaWYgbm9kZSBlbHNlICgoIlMiLCAxKSwpCiAgICAgICAgcmV0dXJuIGNhbmRzW2ludChzZWxmLnJuZy5pbnRlZ2VycygwLCBsZW4oY2FuZHMpKSldCgogICAgZGVmIF9fbGVuX18oc2VsZik6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLm5vZGVzKQo=', 'online_model.py': 'IiIiUGhhc2UgQSBvbmxpbmUgYWN0aW9uLWVmZmVjdCBtb2RlbCAoR3JhcGhSYW5rZXIpIOKAlCBhIHNtYWxsIENOTiB0cmFpbmVkIE9OTElORSwgcGVyIGdhbWUsCnRvIHByZWRpY3Qgd2hpY2ggYWN0aW9ucy9jbGlja3MgY2F1c2UgYSBmcmFtZSBjaGFuZ2UsIHVzZWQgT05MWSB0byByZS1yYW5rIHRoZSBleHBsb3JlcidzCmFscmVhZHktc2FuY3Rpb25lZCBlcXVhbC10aWVyIHRpZS1icmVhayBjYW5kaWRhdGVzIChzZWUgb25saW5lX2V4cGxvcmVyLnB5KS4KCkRlc2lnbiAoZnJvbSByZWJ1aWxkL1BIQVNFX0FfUExBTi5tZCwganVkZ2UtZ2F0ZWQpOgotIElucHV0OiAxNi1jaGFubmVsIG9uZS1ob3QgNjR4NjQgKHBlcmNlcHRpb24uZW5jb2RlX29uZWhvdCkuCi0gQmFja2JvbmU6IDQgY29udiBsYXllcnMgKyBHcm91cE5vcm0gKGJhdGNoLXNpemUtaW5kZXBlbmRlbnQ7IE5PVCBCYXRjaE5vcm0pLgotIEhlYWRzOiBhY3Rpb24tZWZmZWN0IGhlYWQgLT4gUChmcmFtZS1jaGFuZ2UpIGZvciBBQ1RJT04xLTU7IGEgMXgxLWNvbnYgY2xpY2sgaGVhZCAtPiBhCiAgNjR4NjQgcGVyLXBpeGVsIFAoZnJhbWUtY2hhbmdlKSBtYXAgZm9yIEFDVElPTjYgKHNwYXRpYWxseSBmYWl0aGZ1bCwgTk9UIGZsYXR0ZW5lZCkuCi0gT25saW5lIHRyYWluaW5nOiBpbnQ4LWdyaWQgcmVwbGF5IGJ1ZmZlciwgaGFzaC1kZWR1cCBvZiAoc3RhdGUsYWN0aW9uKSwgQkNFIGV2ZXJ5IE4gc3RlcHM7CiAgbGFiZWwgPSB0aGUgZXhwbG9yZXIncyBNQVNLRUQgb2JqZWN0X3N0YXRlX2tleSBjaGFuZ2VkIChub3QgcmF3IHBpeGVscyAtPiBpZ25vcmVzIEhVRCBub2lzZSkuCiAgQnVmZmVyICsgbW9kZWwgcmVzZXQgYmV0d2VlbiBsZXZlbHMuCgpUT1JDSC1PUFRJT05BTCArIEhBUkRFTkVEIEZBSUwtU0FGRSAoZXZhbCBpbWFnZSBtYXkgZ2l2ZSBhIFAxMDAgY2FwLTYuMCB0aGF0IHRvcmNoIGNhbid0IHVzZSwKb3Igbm8gdG9yY2ggYXQgYWxsKTogdGhlIG1vZGVsIGlzIHVzYWJsZSBPTkxZIGlmIHRvcmNoIGltcG9ydHMgQU5EIGEgdGlueSBvcCBvbiB0aGUgY2hvc2VuCmRldmljZSBhY3R1YWxseSBTVUNDRUVEUy4gT3RoZXJ3aXNlIGB1c2FibGVgIGlzIEZhbHNlIGFuZCBldmVyeSBtZXRob2QgaXMgYSBuby1vcCwgc28gdGhlCmNhbGxlciBkZWdyYWRlcyB0byB0aGUgcHVyZSBTYWxpZW5jZUV4cGxvcmVyICh0aGUgMC4zMyBmbG9vcikuIE5vdGhpbmcgaGVyZSBldmVyIHJhaXNlcyB0bwp0aGUgY2FsbGVyLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4gaW1wb3J0IHBlcmNlcHRpb24gYXMgUAoKTlVNX0NPTE9SUyA9IDE2ClNJTVBMRV9JRFMgPSBbMSwgMiwgMywgNCwgNV0KCgpkZWYgX3NlbGVjdF9kZXZpY2UoYWxsb3dfY3B1OiBib29sID0gRmFsc2UpOgogICAgIiIiUmV0dXJuICh0b3JjaCwgZGV2aWNlKSBpZiBhIFVTQUJMRSBjb21wdXRlIGRldmljZSBleGlzdHMsIGVsc2UgKHRvcmNoX29yX05vbmUsIE5vbmUpLgoKICAgIEhhcmRlbmVkIEdQVSBnYXRlOiBjdWRhLmlzX2F2YWlsYWJsZSgpIGlzIE5PVCB0cnVzdGVkIGFsb25lIOKAlCB3ZSBydW4gYSB0aW55IG1hdG11bCBvbiB0aGUKICAgIEdQVSBhbmQgcmVxdWlyZSBpdCB0byBTVUNDRUVEIChhIFAxMDAvY2FwLTYuMCB3aXRoIGFuIGluY29tcGF0aWJsZSB0b3JjaCBidWlsZCByZXBvcnRzCiAgICBhdmFpbGFibGUgYnV0IGVycm9ycyBvbiBvcHMpLiBHUFUtT05MWSBieSBkZWZhdWx0OiBpZiBubyB1c2FibGUgQ1VEQSBkZXZpY2UsIHJldHVybiBOb25lIHNvCiAgICB0aGUgbW9kZWwgZGlzYWJsZXMgYW5kIHRoZSBhZ2VudCBydW5zIHRoZSBwdXJlIFNhbGllbmNlRXhwbG9yZXIgYXQgZnVsbCBzcGVlZCDigJQgYmVjYXVzZSBhdAogICAgZXZhbCB0aGUgYnVkZ2V0IGlzIFdBTEwtQ0xPQ0sgKDEyaCkgYW5kIENQVSB0cmFpbmluZyBtZWFucyBmZXdlciBhY3Rpb25zID0gYSBuZXQgcmVncmVzc2lvbi4KICAgIGFsbG93X2NwdT1UcnVlIChsb2NhbCBkZXYgb25seSkgcGVybWl0cyBhIENQVSBkZXZpY2UgdG8gdmFsaWRhdGUgdGhlIG1vZGVsIGxvZ2ljLgogICAgIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICB0cnk6CiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgZGV2ID0gdG9yY2guZGV2aWNlKCJjdWRhOjAiKQogICAgICAgICAgICBfeCA9IHRvcmNoLnplcm9zKCg4LCA4KSwgZGV2aWNlPWRldikKICAgICAgICAgICAgZmxvYXQoKF94IEAgX3gpLnN1bSgpLml0ZW0oKSkgICMgZm9yY2VzIGEgcmVhbCBrZXJuZWwgbGF1bmNoCiAgICAgICAgICAgIHJldHVybiB0b3JjaCwgZGV2CiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIGlmIGFsbG93X2NwdToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGRldiA9IHRvcmNoLmRldmljZSgiY3B1IikKICAgICAgICAgICAgX3ggPSB0b3JjaC56ZXJvcygoNCwgNCksIGRldmljZT1kZXYpCiAgICAgICAgICAgIGZsb2F0KChfeCBAIF94KS5zdW0oKS5pdGVtKCkpCiAgICAgICAgICAgIHJldHVybiB0b3JjaCwgZGV2CiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLCBOb25lCiAgICByZXR1cm4gdG9yY2gsIE5vbmUgICMgR1BVLW9ubHk6IG5vIHVzYWJsZSBDVURBIC0+IGRpc2FibGVkIChwdXJlIFNhbGllbmNlRXhwbG9yZXIpCgoKZGVmIF9idWlsZF9uZXQodG9yY2gsIGRldmljZSk6CiAgICBubiA9IHRvcmNoLm5uCgogICAgY2xhc3MgX05ldChubi5Nb2R1bGUpOgogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgICAgIGRlZiBibG9jayhjaSwgY28pOgogICAgICAgICAgICAgICAgcmV0dXJuIG5uLlNlcXVlbnRpYWwobm4uQ29udjJkKGNpLCBjbywgMywgcGFkZGluZz0xKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5uLkdyb3VwTm9ybSg4LCBjbyksIG5uLlJlTFUoKSkKICAgICAgICAgICAgc2VsZi5iYWNrYm9uZSA9IG5uLlNlcXVlbnRpYWwoYmxvY2soTlVNX0NPTE9SUywgMzIpLCBibG9jaygzMiwgNDgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBibG9jayg0OCwgNjQpLCBibG9jayg2NCwgNjQpKQogICAgICAgICAgICBzZWxmLmFjdGlvbl9oZWFkID0gbm4uU2VxdWVudGlhbChubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgxKSwgbm4uRmxhdHRlbigpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBubi5MaW5lYXIoNjQsIGxlbihTSU1QTEVfSURTKSkpCiAgICAgICAgICAgIHNlbGYuY2xpY2tfaGVhZCA9IG5uLkNvbnYyZCg2NCwgMSwgMSkgICMgMXgxIGNvbnYgLT4gcGVyLXBpeGVsIGxvZ2l0IG1hcAoKICAgICAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICAgICAgZiA9IHNlbGYuYmFja2JvbmUoeCkKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYWN0aW9uX2hlYWQoZiksIHNlbGYuY2xpY2tfaGVhZChmKS5zcXVlZXplKDEpICAjIChCLDUpLCAoQixILFcpCgogICAgcmV0dXJuIF9OZXQoKS50byhkZXZpY2UpCgoKY2xhc3MgT25saW5lQWN0aW9uRWZmZWN0TW9kZWw6CiAgICAiIiJPbmxpbmUgZnJhbWUtY2hhbmdlIHByZWRpY3Rvci4gYHVzYWJsZWAgZ2F0ZXMgZXZlcnl0aGluZzsgbm8gbWV0aG9kIGV2ZXIgcmFpc2VzLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzZWVkOiBpbnQgPSAwLCB0cmFpbl9ldmVyeTogaW50ID0gOCwgbWF4X2J1ZmZlcjogaW50ID0gMzAwMDAsCiAgICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZTogaW50ID0gNjQsIGNvbmZfdGhyZXNob2xkOiBmbG9hdCA9IDAuMCwKICAgICAgICAgICAgICAgICBhbGxvd19jcHU6IGJvb2wgPSBGYWxzZSkgLT4gTm9uZToKICAgICAgICBzZWxmLnRyYWluX2V2ZXJ5ID0gdHJhaW5fZXZlcnkKICAgICAgICBzZWxmLm1heF9idWZmZXIgPSBtYXhfYnVmZmVyCiAgICAgICAgc2VsZi5iYXRjaF9zaXplID0gYmF0Y2hfc2l6ZQogICAgICAgIHNlbGYuY29uZl90aHJlc2hvbGQgPSBjb25mX3RocmVzaG9sZAogICAgICAgIHNlbGYudXNhYmxlID0gRmFsc2UKICAgICAgICBzZWxmLmRldmljZV9raW5kID0gIm5vbmUiCiAgICAgICAgc2VsZi5fdG9yY2ggPSBOb25lCiAgICAgICAgc2VsZi5fZGV2ID0gTm9uZQogICAgICAgIHNlbGYuX25ldCA9IE5vbmUKICAgICAgICBzZWxmLl9vcHQgPSBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICB0b3JjaCwgZGV2ID0gX3NlbGVjdF9kZXZpY2UoYWxsb3dfY3B1PWFsbG93X2NwdSkKICAgICAgICAgICAgaWYgdG9yY2ggaXMgbm90IE5vbmUgYW5kIGRldiBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICAgICAgICAgICAgICBzZWxmLl90b3JjaCA9IHRvcmNoCiAgICAgICAgICAgICAgICBzZWxmLl9kZXYgPSBkZXYKICAgICAgICAgICAgICAgIHNlbGYuX25ldCA9IF9idWlsZF9uZXQodG9yY2gsIGRldikKICAgICAgICAgICAgICAgIHNlbGYuX29wdCA9IHRvcmNoLm9wdGltLkFkYW0oc2VsZi5fbmV0LnBhcmFtZXRlcnMoKSwgbHI9MWUtMykKICAgICAgICAgICAgICAgIHNlbGYudXNhYmxlID0gVHJ1ZQogICAgICAgICAgICAgICAgc2VsZi5kZXZpY2Vfa2luZCA9IGRldi50eXBlCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgc2VsZi51c2FibGUgPSBGYWxzZQogICAgICAgIHNlbGYuX2J1ZjogZGljdFtieXRlcywgdHVwbGVdID0ge30gICMgKHN0YXRlX2J5dGVzLGFjdGlvbl9pZCx4LHkpIC0+IChncmlkX2ludDgsIGxhYmVsKQogICAgICAgIHNlbGYuX3N0ZXBzID0gMAoKICAgICMgLS0tIGxpZmVjeWNsZSAtLS0KICAgIGRlZiByZXNldF9sZXZlbChzZWxmKToKICAgICAgICAiIiJSZXNldCBidWZmZXIgKyBtb2RlbCBiZXR3ZWVuIGxldmVscyAoU3RvY2hhc3RpY0dvb3NlIHJlY2lwZSkuIiIiCiAgICAgICAgaWYgbm90IHNlbGYudXNhYmxlOgogICAgICAgICAgICByZXR1cm4KICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYuX25ldCA9IF9idWlsZF9uZXQoc2VsZi5fdG9yY2gsIHNlbGYuX2RldikKICAgICAgICAgICAgc2VsZi5fb3B0ID0gc2VsZi5fdG9yY2gub3B0aW0uQWRhbShzZWxmLl9uZXQucGFyYW1ldGVycygpLCBscj0xZS0zKQogICAgICAgICAgICBzZWxmLl9idWYuY2xlYXIoKQogICAgICAgICAgICBzZWxmLl9zdGVwcyA9IDAKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBzZWxmLnVzYWJsZSA9IEZhbHNlCgogICAgIyAtLS0gdHJhaW5pbmcgZGF0YSAtLS0KICAgIGRlZiBvYnNlcnZlKHNlbGYsIGdyaWRfYmVmb3JlLCBhY3Rpb24sIGNoYW5nZWQ6IGJvb2wpOgogICAgICAgICIiIlJlY29yZCBhIChzdGF0ZSwgYWN0aW9uKSAtPiBmcmFtZS1jaGFuZ2VkPyBzYW1wbGUgKGhhc2gtZGVkdXBlZCwgbGF0ZXN0LXdpbnMpLiIiIgogICAgICAgIGlmIG5vdCBzZWxmLnVzYWJsZToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBrZXkgPSBfc2FtcGxlX2tleShncmlkX2JlZm9yZSwgYWN0aW9uKQogICAgICAgICAgICBzZWxmLl9idWZba2V5XSA9IChucC5hc2FycmF5KGdyaWRfYmVmb3JlLCBkdHlwZT1ucC5pbnQ4KSwgMS4wIGlmIGNoYW5nZWQgZWxzZSAwLjApCiAgICAgICAgICAgIGlmIGxlbihzZWxmLl9idWYpID4gc2VsZi5tYXhfYnVmZmVyOgogICAgICAgICAgICAgICAgIyBkcm9wIGFuIGFyYml0cmFyeSBvbGRlc3QtaXNoIGVudHJ5IChkaWN0IHByZXNlcnZlcyBpbnNlcnRpb24gb3JkZXIpCiAgICAgICAgICAgICAgICBzZWxmLl9idWYucG9wKG5leHQoaXRlcihzZWxmLl9idWYpKSkKICAgICAgICAgICAgc2VsZi5fc3RlcHMgKz0gMQogICAgICAgICAgICBpZiBzZWxmLl9zdGVwcyAlIHNlbGYudHJhaW5fZXZlcnkgPT0gMDoKICAgICAgICAgICAgICAgIHNlbGYuX3RyYWluX3N0ZXAoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHNlbGYudXNhYmxlID0gRmFsc2UKCiAgICBkZWYgX3RyYWluX3N0ZXAoc2VsZik6CiAgICAgICAgaWYgbm90IHNlbGYudXNhYmxlIG9yIGxlbihzZWxmLl9idWYpIDwgODoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdG9yY2ggPSBzZWxmLl90b3JjaAogICAgICAgIHRyeToKICAgICAgICAgICAgaXRlbXMgPSBsaXN0KHNlbGYuX2J1Zi5pdGVtcygpKQogICAgICAgICAgICBpZHggPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VsZi5fc3RlcHMpLmludGVnZXJzKDAsIGxlbihpdGVtcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2l6ZT1taW4oc2VsZi5iYXRjaF9zaXplLCBsZW4oaXRlbXMpKSkKICAgICAgICAgICAgZ3JpZHMsIGFfaWR4LCB4cywgeXMsIGxhYmVscywgaXNfY2xpY2sgPSBbXSwgW10sIFtdLCBbXSwgW10sIFtdCiAgICAgICAgICAgIGZvciBqIGluIGlkeDoKICAgICAgICAgICAgICAgIChzYiwgYWlkLCB4LCB5KSwgKGcsIGxhYikgPSBpdGVtc1tpbnQoaildCiAgICAgICAgICAgICAgICBncmlkcy5hcHBlbmQoUC5lbmNvZGVfb25laG90KGcpKQogICAgICAgICAgICAgICAgbGFiZWxzLmFwcGVuZChsYWIpCiAgICAgICAgICAgICAgICBpZiBhaWQgPT0gNjoKICAgICAgICAgICAgICAgICAgICBpc19jbGljay5hcHBlbmQoVHJ1ZSk7IHhzLmFwcGVuZCh4KTsgeXMuYXBwZW5kKHkpOyBhX2lkeC5hcHBlbmQoMCkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgaXNfY2xpY2suYXBwZW5kKEZhbHNlKTsgeHMuYXBwZW5kKDApOyB5cy5hcHBlbmQoMCk7IGFfaWR4LmFwcGVuZChTSU1QTEVfSURTLmluZGV4KGFpZCkpCiAgICAgICAgICAgIFggPSB0b3JjaC5hc190ZW5zb3IobnAuc3RhY2soZ3JpZHMpLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9c2VsZi5fZGV2KQogICAgICAgICAgICBsYWIgPSB0b3JjaC5hc190ZW5zb3IobGFiZWxzLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9c2VsZi5fZGV2KQogICAgICAgICAgICBhY3RfbG9naXRzLCBjbGlja19tYXAgPSBzZWxmLl9uZXQoWCkKICAgICAgICAgICAgcHJlZHMgPSBbXQogICAgICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oaWR4KSk6CiAgICAgICAgICAgICAgICBpZiBpc19jbGlja1tpXToKICAgICAgICAgICAgICAgICAgICBwcmVkcy5hcHBlbmQoY2xpY2tfbWFwW2ksIHlzW2ldLCB4c1tpXV0pCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHByZWRzLmFwcGVuZChhY3RfbG9naXRzW2ksIGFfaWR4W2ldXSkKICAgICAgICAgICAgcHJlZCA9IHRvcmNoLnN0YWNrKHByZWRzKQogICAgICAgICAgICBsb3NzID0gdG9yY2gubm4uZnVuY3Rpb25hbC5iaW5hcnlfY3Jvc3NfZW50cm9weV93aXRoX2xvZ2l0cyhwcmVkLCBsYWIpCiAgICAgICAgICAgIHNlbGYuX29wdC56ZXJvX2dyYWQoKTsgbG9zcy5iYWNrd2FyZCgpOyBzZWxmLl9vcHQuc3RlcCgpCiAgICAgICAgICAgIHJldHVybiBmbG9hdChsb3NzLml0ZW0oKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBzZWxmLnVzYWJsZSA9IEZhbHNlCiAgICAgICAgICAgIHJldHVybiBOb25lCgogICAgIyAtLS0gaW5mZXJlbmNlICh0aGUgcmUtcmFua2VyKSAtLS0KICAgIGRlZiBiZXN0KHNlbGYsIGdyaWQsIGNob2ljZXMpOgogICAgICAgICIiIlJldHVybiB0aGUgY2FuZGlkYXRlIGluIGBjaG9pY2VzYCB3aXRoIHRoZSBoaWdoZXN0IHByZWRpY3RlZCBmcmFtZS1jaGFuZ2UgcHJvYiwKICAgICAgICBvciBOb25lIGlmIG5vdCB1c2FibGUgLyBub3QgY29uZmlkZW50IC8gZXJyb3IgKGNhbGxlciB0aGVuIGtlZXBzIGl0cyBvd24gcGljaykuCiAgICAgICAgYGNob2ljZXNgIGFyZSBhY3Rpb24gdHVwbGVzICgiUyIsYWlkKSB8ICgiQyIseCx5KS4iIiIKICAgICAgICBpZiBub3Qgc2VsZi51c2FibGUgb3Igc2VsZi5fc3RlcHMgPCBzZWxmLnRyYWluX2V2ZXJ5IG9yIGxlbihjaG9pY2VzKSA8IDI6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgdG9yY2ggPSBzZWxmLl90b3JjaAogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICAgICBYID0gdG9yY2guYXNfdGVuc29yKFAuZW5jb2RlX29uZWhvdChncmlkKVtOb25lXSwgZHR5cGU9dG9yY2guZmxvYXQzMiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPXNlbGYuX2RldikKICAgICAgICAgICAgICAgIGFjdF9sb2dpdHMsIGNsaWNrX21hcCA9IHNlbGYuX25ldChYKQogICAgICAgICAgICAgICAgYWN0X3AgPSB0b3JjaC5zaWdtb2lkKGFjdF9sb2dpdHNbMF0pCiAgICAgICAgICAgICAgICBjbGlja19wID0gdG9yY2guc2lnbW9pZChjbGlja19tYXBbMF0pCiAgICAgICAgICAgICAgICBiZXN0X2MsIGJlc3RfdiA9IE5vbmUsIC0xLjAKICAgICAgICAgICAgICAgIGZvciBjIGluIGNob2ljZXM6CiAgICAgICAgICAgICAgICAgICAgaWYgY1swXSA9PSAiUyI6CiAgICAgICAgICAgICAgICAgICAgICAgIHYgPSBmbG9hdChhY3RfcFtTSU1QTEVfSURTLmluZGV4KGNbMV0pXSkgaWYgY1sxXSBpbiBTSU1QTEVfSURTIGVsc2UgMC4wCiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgdiA9IGZsb2F0KGNsaWNrX3BbaW50KGNbMl0pLCBpbnQoY1sxXSldKSAgIyBtYXBbeSwgeF0KICAgICAgICAgICAgICAgICAgICBpZiB2ID4gYmVzdF92OgogICAgICAgICAgICAgICAgICAgICAgICBiZXN0X3YsIGJlc3RfYyA9IHYsIGMKICAgICAgICAgICAgaWYgYmVzdF92IDwgc2VsZi5jb25mX3RocmVzaG9sZDoKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgICAgIHJldHVybiBiZXN0X2MKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBzZWxmLnVzYWJsZSA9IEZhbHNlCiAgICAgICAgICAgIHJldHVybiBOb25lCgoKZGVmIF9zYW1wbGVfa2V5KGdyaWQsIGFjdGlvbik6CiAgICBzYiA9IG5wLmFzYXJyYXkoZ3JpZCwgZHR5cGU9bnAuaW50OCkudG9ieXRlcygpCiAgICBpZiBhY3Rpb25bMF0gPT0gIlMiOgogICAgICAgIHJldHVybiAoc2IsIGFjdGlvblsxXSwgMCwgMCkKICAgIHJldHVybiAoc2IsIDYsIGludChhY3Rpb25bMV0pLCBpbnQoYWN0aW9uWzJdKSkK', 'online_explorer.py': 'IiIiUGhhc2UgQSBHcmFwaFJhbmtlciDigJQgT25saW5lTGVhcm5pbmdFeHBsb3JlciBjb21wb3NlcyAobmV2ZXIgbW9kaWZpZXMpIFNhbGllbmNlRXhwbG9yZXIuCgpUaGUgZXhwbG9yZXIncyBvYmplY3QtZ3JhcGgsIEJGUy10by1mcm9udGllciwgc3VzcGljaW9uIGZpbHRlciwgSFVEIG1hc2ssIGV4cGxvaXQtcmV3YXJkLCBhbmQKcmVzZXQgbG9naWMgcmVtYWluIHRoZSBzb2xlIGV4ZWN1dGlvbiBiYWNrYm9uZSBhbmQgbWVtb3J5LiBBbiBvbmxpbmUgcGVyLWdhbWUgQ05OCihvbmxpbmVfbW9kZWwuT25saW5lQWN0aW9uRWZmZWN0TW9kZWwpIGRvZXMgZXhhY3RseSBPTkUgdGhpbmc6IHdoZW4gdGhlIGV4cGxvcmVyJ3MgcGljayBpcyBhCnByb3ZhYmxlIGVxdWFsLXRpZXIgUkFORE9NIHRpZS1icmVhayBhbW9uZyA+MSB1bnRyaWVkIGNhbmRpZGF0ZXMsIHRoZSBtb2RlbCByZS1vcmRlcnMgdGhhdAoqYWxyZWFkeS1zYW5jdGlvbmVkKiBzZXQgYnkgcHJlZGljdGVkIGZyYW1lLWNoYW5nZSBwcm9iYWJpbGl0eSBhbmQgc3Vic3RpdHV0ZXMgaXRzIHRvcCBwaWNrLgoKSXQgbmV2ZXIgcGlja3MgYSB0aWVyLCBuZXZlciBvdmVycmlkZXMgcmV3YXJkX2FjdGlvbiAvIGEgcGxhbiByZXBsYXkgLyBhIHJlc2V0IC8gQkZTLCBuZXZlcgppbnZlbnRzIGEgY2FuZGlkYXRlLCBuZXZlciBwbGFucyBvdmVyIHByZWRpY3Rpb25zLiBPbiBBTlkgbW9kZWwgZXJyb3Igb3Igd2hlbiB0aGUgbW9kZWwgaXMgbm90CnVzYWJsZSAobm8gdG9yY2ggLyB1bnVzYWJsZSBHUFUgLyB1bnRyYWluZWQgLyBub3QgY29uZmlkZW50KSBpdCByZXR1cm5zIHRoZSBiYXNlIHRva2VuIHZlcmJhdGltCi0+IHRoZSBhZ2VudCBkZWdyYWRlcyB0byB0aGUgcHVyZSBTYWxpZW5jZUV4cGxvcmVyICh0aGUgYmFua2VkIDAuMzMgZmxvb3IpLiBTZWFtIG1pcnJvcnMKd21fcG9saWN5LnB5IChiYXNlLmRlY2lkZSBmaXJzdDsgd3JpdGUgYmFjayBiYXNlLnByZXZfYWN0aW9uIG9uIG92ZXJyaWRlKS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuc2FsaWVuY2VfZXhwbG9yZXIgaW1wb3J0IFNhbGllbmNlRXhwbG9yZXIKZnJvbSAub25saW5lX21vZGVsIGltcG9ydCBPbmxpbmVBY3Rpb25FZmZlY3RNb2RlbAoKCmNsYXNzIE9ubGluZUxlYXJuaW5nRXhwbG9yZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgc2VlZDogaW50ID0gMCwgdHJ1c3RfdGhyZXNob2xkOiBpbnQgPSAzLCBib3JkZXJfbWFzazogaW50ID0gMiwKICAgICAgICAgICAgICAgICBjb25mX3RocmVzaG9sZDogZmxvYXQgPSAwLjU1LCBicmVha2VyX2xpbWl0OiBpbnQgPSAxMiwKICAgICAgICAgICAgICAgICBhbGxvd19jcHU6IGJvb2wgPSBGYWxzZSwgKiptb2RlbF9rdykgLT4gTm9uZToKICAgICAgICBzZWxmLmJhc2UgPSBTYWxpZW5jZUV4cGxvcmVyKHNlZWQ9c2VlZCwgdHJ1c3RfdGhyZXNob2xkPXRydXN0X3RocmVzaG9sZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJvcmRlcl9tYXNrPWJvcmRlcl9tYXNrKQogICAgICAgIHNlbGYubW9kZWwgPSBPbmxpbmVBY3Rpb25FZmZlY3RNb2RlbChzZWVkPXNlZWQsIGNvbmZfdGhyZXNob2xkPWNvbmZfdGhyZXNob2xkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhbGxvd19jcHU9YWxsb3dfY3B1LCAqKm1vZGVsX2t3KQogICAgICAgIHNlbGYuYnJlYWtlcl9saW1pdCA9IGJyZWFrZXJfbGltaXQKICAgICAgICBzZWxmLl9sYXN0X2dyaWQgPSBOb25lCiAgICAgICAgc2VsZi5fbGFzdF9hY3Rpb24gPSBOb25lICAgICAgICMgYWN0aW9uIGFjdHVhbGx5IGV4ZWN1dGVkIGludG8gdGhlIGxhc3QgZnJhbWUKICAgICAgICBzZWxmLl9sYXN0X2tleSA9IE5vbmUgICAgICAgICAgIyBiYXNlIGtleSB0aGUgbGFzdCBhY3Rpb24gd2FzIHRha2VuIGZyb20KICAgICAgICBzZWxmLl9sYXN0X3dhc19vdmVycmlkZSA9IEZhbHNlCiAgICAgICAgc2VsZi5fbGV2ZWwgPSAwCiAgICAgICAgc2VsZi5fbGV2ZWxfZGlzYWJsZWQgPSBGYWxzZQogICAgICAgIHNlbGYuX25vX2NoYW5nZV9vdmVycmlkZXMgPSAwCiAgICAgICAgIyB0ZWxlbWV0cnkgKEEuMSByZWR1bmRhbmN5IHByb29mIC8gQS40IGFuYWx5c2lzKQogICAgICAgIHNlbGYubl9vdmVycmlkZXMgPSAwCiAgICAgICAgc2VsZi5uX3RpZWJyZWFrcyA9IDAKCiAgICAjIGR1Y2stdHlwZSB0aGUgcnVubmVyJ3Mgc3RhdGVzX3NlZW4gKHBvbC5ncy53bSkKICAgIEBwcm9wZXJ0eQogICAgZGVmIGdzKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLmJhc2UuZ3MKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYuYmFzZSkKCiAgICBkZWYgZGVjaWRlKHNlbGYsIGdyaWQsIGdzdGF0ZV90ZXJtaW5hbCwgZ3N0YXRlX25vdHBsYXllZCwgbGV2ZWxzLCBhdmFpbGFibGUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgZ3JpZCA9IG5wLmFzYXJyYXkoZ3JpZCwgZHR5cGU9bnAuaW50OCkgICMgbWF0Y2ggZW5naW5lIGZyYW1lczsgZGVmZW5kcyB3cm9uZyBkdHlwZQogICAgICAgICAgICByZXR1cm4gc2VsZi5fZGVjaWRlKGdyaWQsIGdzdGF0ZV90ZXJtaW5hbCwgZ3N0YXRlX25vdHBsYXllZCwgbGV2ZWxzLCBhdmFpbGFibGUpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgIyBhYnNvbHV0ZSBmYWlsLXNhZmUgKEFDLTQpOiBuZXZlciByYWlzZTsgYWN0IHNhZmVseS4KICAgICAgICAgICAgaWYgZ3N0YXRlX3Rlcm1pbmFsIG9yIGdzdGF0ZV9ub3RwbGF5ZWQ6CiAgICAgICAgICAgICAgICByZXR1cm4gKCJyZXNldCIsKQogICAgICAgICAgICBmb3IgYSBpbiAoYXZhaWxhYmxlIG9yIFsxLCAyLCAzLCA0LCA1XSk6CiAgICAgICAgICAgICAgICBpZiBhICE9IDY6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuICgiUyIsIGludChhKSkKICAgICAgICAgICAgcmV0dXJuICgiUyIsIDEpCgogICAgZGVmIF9kZWNpZGUoc2VsZiwgZ3JpZCwgZ3N0YXRlX3Rlcm1pbmFsLCBnc3RhdGVfbm90cGxheWVkLCBsZXZlbHMsIGF2YWlsYWJsZSk6CiAgICAgICAgIyBzbmFwc2hvdCB0aGUgY2F1c2Utc3RhdGUgQkVGT1JFIGJhc2UgbXV0YXRlcyBpdAogICAgICAgIGtleV9wcmV2ID0gc2VsZi5iYXNlLnByZXZfa2V5CiAgICAgICAgYWN0aW9uX3ByZXYgPSBzZWxmLl9sYXN0X2FjdGlvbgogICAgICAgIHBsYW5fZW1wdHlfYmVmb3JlID0gbm90IHNlbGYuYmFzZS5wbGFuCgogICAgICAgIGJhc2VfdG9rZW4gPSBzZWxmLmJhc2UuZGVjaWRlKGdyaWQsIGdzdGF0ZV90ZXJtaW5hbCwgZ3N0YXRlX25vdHBsYXllZCwgbGV2ZWxzLCBhdmFpbGFibGUpCgogICAgICAgICMgdGVybWluYWwvbm90cGxheWVkL3Jlc2V0OiBwYXNzdGhyb3VnaDsgZHJvcCBjcm9zcy1lcGlzb2RlIHRyYWluaW5nIGxpbmsKICAgICAgICBpZiBnc3RhdGVfdGVybWluYWwgb3IgZ3N0YXRlX25vdHBsYXllZCBvciBiYXNlX3Rva2VuWzBdID09ICJyZXNldCI6CiAgICAgICAgICAgIHNlbGYuX2xhc3RfZ3JpZCA9IE5vbmUKICAgICAgICAgICAgc2VsZi5fbGFzdF9hY3Rpb24gPSBOb25lCiAgICAgICAgICAgIHNlbGYuX2xhc3Rfa2V5ID0gTm9uZQogICAgICAgICAgICBzZWxmLl9sYXN0X3dhc19vdmVycmlkZSA9IEZhbHNlCiAgICAgICAgICAgIHJldHVybiBiYXNlX3Rva2VuCgogICAgICAgICMgcGVyLWxldmVsIHJlc2V0IChtaXJyb3I6IG1vZGVsICsgYnVmZmVyIHJlc2V0IGJldHdlZW4gbGV2ZWxzKQogICAgICAgIGlmIGxldmVscyAhPSBzZWxmLl9sZXZlbDoKICAgICAgICAgICAgc2VsZi5fbGV2ZWwgPSBsZXZlbHMKICAgICAgICAgICAgc2VsZi5fbGV2ZWxfZGlzYWJsZWQgPSBGYWxzZQogICAgICAgICAgICBzZWxmLl9ub19jaGFuZ2Vfb3ZlcnJpZGVzID0gMAogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzZWxmLm1vZGVsLnJlc2V0X2xldmVsKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgY3VyX2tleSA9IHNlbGYuYmFzZS5wcmV2X2tleSAgIyBiYXNlIGp1c3Qgc2V0IHRoaXMgdG8ga2V5KGdyaWQpCgogICAgICAgICMgdHJhaW4gb24gdGhlIFBSRVZJT1VTIGFjdGlvbidzIG9ic2VydmVkIG91dGNvbWUgKGNoYW5nZWQgPSBtYXNrZWQga2V5IGNoYW5nZWQpCiAgICAgICAgaWYgKGFjdGlvbl9wcmV2IGlzIG5vdCBOb25lIGFuZCBrZXlfcHJldiBpcyBub3QgTm9uZSBhbmQgc2VsZi5fbGFzdF9ncmlkIGlzIG5vdCBOb25lKToKICAgICAgICAgICAgY2hhbmdlZCA9IGN1cl9rZXkgIT0ga2V5X3ByZXYKICAgICAgICAgICAgaWYgc2VsZi5fbGFzdF93YXNfb3ZlcnJpZGUgYW5kIG5vdCBjaGFuZ2VkOgogICAgICAgICAgICAgICAgc2VsZi5fbm9fY2hhbmdlX292ZXJyaWRlcyArPSAxCiAgICAgICAgICAgICAgICBpZiBzZWxmLl9ub19jaGFuZ2Vfb3ZlcnJpZGVzID49IHNlbGYuYnJlYWtlcl9saW1pdDoKICAgICAgICAgICAgICAgICAgICBzZWxmLl9sZXZlbF9kaXNhYmxlZCA9IFRydWUKICAgICAgICAgICAgZWxpZiBjaGFuZ2VkOgogICAgICAgICAgICAgICAgc2VsZi5fbm9fY2hhbmdlX292ZXJyaWRlcyA9IDAKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5tb2RlbC5vYnNlcnZlKHNlbGYuX2xhc3RfZ3JpZCwgYWN0aW9uX3ByZXYsIGJvb2woY2hhbmdlZCkpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCgogICAgICAgIG91dCA9IGJhc2VfdG9rZW4KICAgICAgICBpc19vdmVycmlkZSA9IEZhbHNlCiAgICAgICAgaWYgbm90IHNlbGYuX2xldmVsX2Rpc2FibGVkIGFuZCBwbGFuX2VtcHR5X2JlZm9yZSBhbmQgbm90IHNlbGYuYmFzZS5wbGFuOgogICAgICAgICAgICBjaG9pY2VzID0gc2VsZi5fdGllYnJlYWtfY2hvaWNlcyhjdXJfa2V5LCBiYXNlX3Rva2VuKQogICAgICAgICAgICBpZiBjaG9pY2VzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgc2VsZi5uX3RpZWJyZWFrcyArPSAxCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgYmVzdCA9IHNlbGYubW9kZWwuYmVzdChncmlkLCBjaG9pY2VzKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBiZXN0ID0gTm9uZQogICAgICAgICAgICAgICAgaWYgYmVzdCBpcyBub3QgTm9uZSBhbmQgYmVzdCAhPSBiYXNlX3Rva2VuOgogICAgICAgICAgICAgICAgICAgIG91dCA9IGJlc3QKICAgICAgICAgICAgICAgICAgICBpc19vdmVycmlkZSA9IFRydWUKICAgICAgICAgICAgICAgICAgICBzZWxmLm5fb3ZlcnJpZGVzICs9IDEKICAgICAgICAgICAgICAgICAgICBzZWxmLmJhc2UucHJldl9hY3Rpb24gPSBiZXN0ICAjIHdyaXRlLWJhY2sgKHRoZSBzZWFtIGNydXgpCgogICAgICAgIHNlbGYuX2xhc3RfZ3JpZCA9IG5wLmFzYXJyYXkoZ3JpZCwgZHR5cGU9bnAuaW50OCkuY29weSgpCiAgICAgICAgc2VsZi5fbGFzdF9hY3Rpb24gPSBvdXQKICAgICAgICBzZWxmLl9sYXN0X2tleSA9IGN1cl9rZXkKICAgICAgICBzZWxmLl9sYXN0X3dhc19vdmVycmlkZSA9IGlzX292ZXJyaWRlCiAgICAgICAgcmV0dXJuIG91dAoKICAgIGRlZiBfdGllYnJlYWtfY2hvaWNlcyhzZWxmLCBjdXJfa2V5LCBiYXNlX3Rva2VuKToKICAgICAgICAiIiJSZXR1cm4gdGhlIGVxdWFsLWxvd2VzdC10aWVyIHVudHJpZWQgY2FuZGlkYXRlIHNldCBJRkYgYmFzZV90b2tlbiB3YXMgYSByYW5kb20KICAgICAgICB0aWUtYnJlYWsgcGljayBhbW9uZyA+MSBvZiB0aGVtIChzdGVwLTMpLCBlbHNlIE5vbmUuIE5vIHJlY29uc3RydWN0aW9uIG9mIF9jaG9vc2UncwogICAgICAgIGJyYW5jaGluZyDigJQganVzdCB2ZXJpZmllcyB0aGUgY29uZGl0aW9ucyBvbiB0aGUgcmV0dXJuZWQgdG9rZW4gKyBub2RlIHN0YXRlLiIiIgogICAgICAgIG5vZGUgPSBzZWxmLmJhc2Uubm9kZXMuZ2V0KGN1cl9rZXkpCiAgICAgICAgaWYgbm9kZSBpcyBOb25lIG9yIG5vZGUucmV3YXJkX2FjdGlvbigpIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGcgPSBzZWxmLmJhc2UuYWN0aXZlX2dyb3VwCiAgICAgICAgbG9jYWwgPSBub2RlLnVudHJpZWRfbGUoZykKICAgICAgICBpZiBub3QgbG9jYWw6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgbXAgPSBtaW4obm9kZS50aWVyLmdldChhLCAwKSBmb3IgYSBpbiBsb2NhbCkKICAgICAgICBjaG9pY2VzID0gW2EgZm9yIGEgaW4gbG9jYWwgaWYgbm9kZS50aWVyLmdldChhLCAwKSA9PSBtcF0KICAgICAgICBpZiBsZW4oY2hvaWNlcykgPCAyIG9yIGJhc2VfdG9rZW4gbm90IGluIGNob2ljZXM6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0dXJuIGNob2ljZXMK'}
for _name, _b in PKG.items():
    pathlib.Path('/kaggle/working/arcagi3', _name).write_bytes(base64.b64decode(_b))
print('wrote arcagi3 package:', sorted(PKG))


In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# ARC-AGI-3 submission agent (ARC Prize 2026) — fail-safe adapter.
#
# Wraps arcagi3.policy.HybridPolicy in the official Agent interface. The arcagi3 package is
# written to /kaggle/working/arcagi3 by the notebook (self-contained) and/or attached as the
# arcagi3-agent dataset; we add every plausible path to sys.path. If the package cannot be
# imported for ANY reason, we fall back to a built-in random policy so the agent ALWAYS acts
# (a scored submission beats an ERROR, and tells us import was the issue).
# =====================================================================
import os
import random
import sys
import time
import traceback

_CANDIDATES = [
    "/kaggle/working",                  # notebook writes arcagi3/ here (primary, self-contained)
    "/kaggle/working/ARC-AGI-3-Agents/agents/templates",
    "/kaggle/input/arcagi3-agent",      # dataset fallback
    "/kaggle/input/arcagi3-agent/src",
    "/kaggle/input/arcagi3",
    os.path.dirname(os.path.abspath(__file__)),
]
for _p in _CANDIDATES:
    if _p and os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

_Policy = None
_IMPORT_ERR = None
# Phase A is opt-in via ARCAGI3_ONLINE=1. The online policy is GPU-ONLY + fail-safe: on a
# P100/no-usable-GPU eval image it disables the model and runs the pure SalienceExplorer (the
# banked 0.33), so enabling it can never ship below the floor. Default OFF until A.4 proves it
# beats 0.33 on real games. Banked v6 = SalienceExplorer (border=2, trust=3) = 0.33.
if os.getenv("ARCAGI3_ONLINE") == "1":
    try:
        from arcagi3.online_explorer import OnlineLearningExplorer as _Policy
    except Exception:
        _Policy = None
if _Policy is None:
    try:
        from arcagi3.salience_explorer import SalienceExplorer as _Policy
    except Exception:
        try:  # fallback to the hybrid if the new module is unavailable
            from arcagi3.policy import HybridPolicy as _Policy
        except Exception as _e:  # noqa: BLE001
            _IMPORT_ERR = "".join(traceback.format_exception(type(_e), _e, _e.__traceback__))
            print(f"[my_agent] arcagi3 import FAILED -> random fallback.\n{_IMPORT_ERR}", flush=True)
_HybridPolicy = _Policy  # back-compat name used below

try:
    from agents.agent import Agent as _BaseAgent
except Exception:  # local dev / stub
    class _BaseAgent:  # minimal stub
        def __init__(self, *a, **k):
            self.game_id = k.get("game_id", "?")
            self.action_counter = 0
            self.frames = []

from arcengine import GameAction as _GA  # noqa: E402
from arcengine import GameState as _GS  # noqa: E402

_TIME_BUDGET_S = 8 * 3600 - 5 * 60


class MyAgent(_BaseAgent):
    """Hybrid explorer (fail-safe). Falls back to random actions if arcagi3 is unavailable."""

    MAX_ACTIONS = float("inf")

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._pol = _HybridPolicy() if _HybridPolicy is not None else None
        self._t0 = time.time()
        seed = int(time.time() * 1e6) % (2 ** 32 - 1)
        random.seed(seed)

    def is_done(self, frames, latest_frame):
        try:
            if latest_frame.state is _GS.WIN:
                return True
        except Exception:
            pass
        return (time.time() - self._t0) >= _TIME_BUDGET_S

    def _avail_ids(self, latest_frame):
        out = []
        for a in (getattr(latest_frame, "available_actions", None) or []):
            try:
                out.append(a.value if hasattr(a, "value") else int(a))
            except Exception:
                pass
        return out

    def _random_action(self, latest_frame):
        st = getattr(latest_frame, "state", None)
        if st is _GS.NOT_PLAYED or st is _GS.GAME_OVER:
            a = _GA.RESET
            a.reasoning = "reset"
            return a
        avail = self._avail_ids(latest_frame) or [1, 2, 3, 4]
        aid = random.choice(avail)
        if aid == 6:
            a = _GA.ACTION6
            a.set_data({"x": random.randint(0, 63), "y": random.randint(0, 63)})
            a.reasoning = "random click"
            return a
        a = _GA.from_id(aid)
        a.reasoning = "random"
        return a

    def choose_action(self, frames, latest_frame):
        if self._pol is None:
            return self._random_action(latest_frame)
        try:
            import numpy as np

            arr = np.asarray(latest_frame.frame, dtype=np.int8)
            grid = arr[-1] if arr.ndim == 3 else arr
            st = latest_frame.state
            token = self._pol.decide(
                grid,
                gstate_terminal=(st is _GS.GAME_OVER),
                gstate_notplayed=(st is _GS.NOT_PLAYED),
                levels=int(getattr(latest_frame, "levels_completed", 0) or 0),
                available=self._avail_ids(latest_frame),
            )
            if token[0] == "reset":
                a = _GA.RESET
                a.reasoning = "reset"
                return a
            if token[0] == "S":
                a = _GA.from_id(token[1])
                a.reasoning = "hybrid"
                return a
            a = _GA.ACTION6
            a.set_data({"x": int(token[1]), "y": int(token[2])})
            a.reasoning = "hybrid-click"
            return a
        except Exception as e:  # never die mid-game
            print(f"[my_agent] choose_action error -> random: {e}", flush=True)
            return self._random_action(latest_frame)


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # 1) wait for the gateway that serves the private games
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # 2) copy the official agents framework to a writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # 3) drop our agent into the framework templates (it imports arcagi3 from /kaggle/working)
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # 4) minimal agents/__init__.py: register only what we need (avoid heavy template imports)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
''')

    # 5) .env pointing the framework at the gateway (online mode, no local env files)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # 6) play all games; the gateway records the scorecard -> submission
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 0]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
